# Assignment 1 — QANet

**COMP5329 / Deep Learning — University of Sydney, Semester 1 2026**

Run each section in order. Sections 0–1 are one-time setup steps; Sections 2–4 are the main training and evaluation pipeline.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Adjust this path if your repo is stored elsewhere in Drive.
PROJECT_ROOT = "/content/drive/MyDrive/Assignment1_2026"

In [ ]:
# Please switch to the correct branch, fix or submit the changes, and then don't forget to re-run the pull or start a new session.
BRANCH = "dev/Stage-3"
!cd {PROJECT_ROOT} && git fetch origin && git checkout {BRANCH} && git reset --hard origin/{BRANCH}
!cd {PROJECT_ROOT} && git branch --show-current && git log -1 --oneline


# To address the self-referential issue, if you have modified this notebook file, please open the temporary notebook and execute the following command:

# from google.colab import drive
# drive.mount('/content/drive')

# PROJECT_ROOT = "/content/drive/MyDrive/Assignment1_2026"
# BRANCH = "dev/Stage-3"

# !cd {PROJECT_ROOT} && git reset --hard && git fetch --all && git checkout {BRANCH} && git pull origin {BRANCH}
# !cd {PROJECT_ROOT} && git branch --show-current && git log -1 --oneline

In [ ]:
# Install Python dependencies (run once per session)
!pip install -r {PROJECT_ROOT}/requirements.txt -q
!python -m spacy download en

---
## Section 0 — Environment Setup

Mount Google Drive and install dependencies.

In [ ]:
import sys, os

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())

---
## Section 1 — Download Data *(delete before submitting)*

Downloads the pre-built mini dataset (sampled SQuAD v1.1 train + full dev set,
with GloVe vectors filtered to the mini vocabulary) from GitHub Releases into `_data/`.

> **One-time step.** Once `_data/` exists on your Drive, delete this section before submission.

In [ ]:
from Tools.download import download_mini

download_mini(data_dir="_data")

---
## Section 2 — Preprocess Data *(delete before submitting)*

Tokenises the SQuAD JSON files, builds word/char vocabularies from GloVe, and writes padded index tensors to `_data/`.

> **One-time step.** Once `_data/*.npz` exists on your Drive, delete this section before submission. Re-run only if you change `para_limit`, `ques_limit`, or other shape parameters.

In [ ]:
from Tools.preproc import preprocess

preprocess(
    train_file="_data/squad/train-mini.json",
    dev_file="_data/squad/dev-v1.1.json",
    glove_word_file="_data/glove/glove.mini.txt",
    target_dir="_data",
    para_limit=400,
    ques_limit=50,
)

---
## Section 3 — Train

Trains QANet on SQuAD v1.1 and saves the best checkpoint to `_model/model.pt`.

In [ ]:
from TrainTools.train import train

results = train(
    # ── data paths (must match preprocess outputs) ──────────────────────
    train_npz       = "_data/train.npz",
    dev_npz         = "_data/dev.npz",
    word_emb_json   = "_data/word_emb.json",
    char_emb_json   = "_data/char_emb.json",
    train_eval_json = "_data/train_eval.json",
    dev_eval_json   = "_data/dev_eval.json",
    save_dir        = "_model",
    log_dir         = "_log",

    # ── training loop ────────────────────────────────────────────────────
    num_steps  = 60000,
    batch_size = 8,
    seed       = 42,

    # ── vanilla recipe: SGD, no scheduler, NLL loss ───────────────────────
    optimizer_name = "sgd",
    scheduler_name = "none",
    loss_name      = "qa_nll",
)

print(f"Best F1: {results['best_f1']:.4f}  |  Best EM: {results['best_em']:.4f}")

---
## Section 4 — Evaluate

Loads the saved checkpoint and runs inference on the full dev set.

In [ ]:
from EvaluateTools.evaluate import evaluate

metrics = evaluate(
    dev_npz       = "_data/dev.npz",
    word_emb_json = "_data/word_emb.json",
    char_emb_json = "_data/char_emb.json",
    dev_eval_json = "_data/dev_eval.json",
    save_dir      = "_model",
    log_dir       = "_log",
    ckpt_name     = "model.pt",
)

print(f"F1: {metrics['f1']:.4f}  |  EM: {metrics['exact_match']:.4f}  |  Loss: {metrics['loss']:.6f}")

---
## Section 5 — Stage 3: Experimental Investigation

Causal tracing adapted from Meng et al. (NeurIPS 2022) "Locating and Editing Factual Associations in GPT" to analyze QANet's internal mechanisms.

- **H1**: Component-level causal tracing (Conv vs Self-Attention vs FFN across Model Encoder layers)
- **H2**: Context vs Question dual-stream information flow & CQ Attention bottleneck
- **H3**: Pointer layer asymmetric design (M1/M2/M3 role differentiation)

In [ ]:
## Stage 3 — Setup: Load trained model for analysis

!pip install matplotlib seaborn -q

import argparse, json, os, random
import numpy as np
import torch
import torch.nn.functional as F
from Data import SQuADDataset, load_word_char_mats, load_train_dev_eval, make_loader
from Models import QANet

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CKPT_PATH = "_model/model_best.pt"
DEV_NPZ = "_data/dev.npz"
DEV_EVAL_JSON = "_data/dev_eval.json"

# Load checkpoint and rebuild model
ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
config = ckpt["config"]
model_args = argparse.Namespace(**config)

word_mat, char_mat = load_word_char_mats(model_args)
model = QANet(word_mat, char_mat, model_args).to(DEVICE)
state_key = "model_state" if "model_state" in ckpt else "model_state_dict"
model.load_state_dict(ckpt[state_key])
model.eval()

dataset = SQuADDataset(DEV_NPZ)
import ujson
with open(DEV_EVAL_JSON) as f:
    eval_file = ujson.load(f)

num_params = sum(p.numel() for p in model.parameters())
num_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"{'='*60}")
print(f"Stage 3 — Model & Data Summary")
print(f"{'='*60}")
print(f"Device:           {DEVICE}")
print(f"Checkpoint:       {CKPT_PATH}  (key: '{state_key}')")
print(f"Dev set:          {len(dataset)} samples")
print(f"Model params:     {num_params:,} total, {num_trainable:,} trainable")
print(f"Checkpoint F1:    {ckpt.get('best_f1', '?')}")
print(f"Checkpoint EM:    {ckpt.get('best_em', '?')}")
print(f"Config summary:   d_model={config.get('d_model','?')}, "
      f"num_head={config.get('num_head','?')}, "
      f"emb_enc_blocks=1×{config.get('emb_enc_conv_num', config.get('conv_num','?'))} conv, "
      f"model_enc_blocks=7×2 conv×3 passes")
print(f"{'='*60}")

### H1: Conv vs Self-Attention Causal Importance — Three-Method Cross-Validation

**Hypothesis**: Conv sub-layers are causally more important than Self-Attention for span prediction, concentrated in shallow blocks and early passes.

**Theoretical basis**: QANet paper Table 5 ablation (Conv removal -2.7 F1 > Attn removal -1.3 F1). We extend this with fine-grained analysis.

**Three methods**:
- **Method 2** (below): ROME-style causal tracing — corrupt input, restore individual sub-layers, measure AIE
- **Method 1**: Eval-time global ablation — skip all Conv vs all Attn, measure F1/EM
- **Method 3**: Per-block ablation — skip Conv/Attn in specific (pass, block), measure F1 drop

In [ ]:
from experiments.tracer import (
    qanet_forward, compute_span_prob, compute_start_prob, compute_end_prob,
    build_model_enc_specs, RestoreSpec,
)
from tqdm.auto import tqdm

NUM_SAMPLES_H1 = 200       # reduce if slow
NOISE_STD_SCALE = 3.0
NOISE_REPEATS = 3
MIN_CLEAN_PROB = 0.01
SEED = 42

random.seed(SEED); torch.manual_seed(SEED)
loader_h1 = make_loader(dataset, batch_size=1, shuffle=True)
specs_h1 = build_model_enc_specs()
spec_keys_h1 = [f"p{s.pass_idx}_b{s.block_idx}_{s.component}" for s in specs_h1]

h1_ie = {k: [] for k in spec_keys_h1}
h1_ie_p1 = {k: [] for k in spec_keys_h1}
h1_ie_p2 = {k: [] for k in spec_keys_h1}
h1_te_list = []
samples_used = 0

pbar = tqdm(total=NUM_SAMPLES_H1, desc="H1 Causal Tracing")
with torch.no_grad():
    for batch_idx, (Cwid, Ccid, Qwid, Qcid, y1, y2, ids) in enumerate(loader_h1):
        if samples_used >= NUM_SAMPLES_H1:
            break
        Cwid, Ccid = Cwid.to(DEVICE), Ccid.to(DEVICE)
        Qwid, Qcid = Qwid.to(DEVICE), Qcid.to(DEVICE)
        y1, y2 = y1.to(DEVICE), y2.to(DEVICE)

        # Clean run
        p1_c, p2_c, clean_acts, _ = qanet_forward(model, Cwid, Ccid, Qwid, Qcid, collect=True)
        prob_clean = compute_span_prob(p1_c, p2_c, y1, y2)
        if prob_clean.item() < MIN_CLEAN_PROB:
            continue

        rep_ie = {k: [] for k in spec_keys_h1}
        rep_ie_p1 = {k: [] for k in spec_keys_h1}
        rep_ie_p2 = {k: [] for k in spec_keys_h1}
        rep_te = []

        for rep in range(NOISE_REPEATS):
            ns = SEED + batch_idx * 100 + rep
            p1_x, p2_x, _, _ = qanet_forward(model, Cwid, Ccid, Qwid, Qcid,
                                              corrupt_target="context", noise_std_scale=NOISE_STD_SCALE, noise_seed=ns)
            prob_corrupt = compute_span_prob(p1_x, p2_x, y1, y2)
            prob_p1_x = compute_start_prob(p1_x, y1)
            prob_p2_x = compute_end_prob(p2_x, y2)
            te = (prob_clean - prob_corrupt).item()
            rep_te.append(te)
            if abs(te) < 1e-6:
                continue

            for spec, key in zip(specs_h1, spec_keys_h1):
                p1_r, p2_r, _, _ = qanet_forward(model, Cwid, Ccid, Qwid, Qcid,
                                                  corrupt_target="context", noise_std_scale=NOISE_STD_SCALE, noise_seed=ns,
                                                  clean_acts=clean_acts, restore_spec=spec)
                prob_r = compute_span_prob(p1_r, p2_r, y1, y2)
                rep_ie[key].append((prob_r - prob_corrupt).item())
                rep_ie_p1[key].append((compute_start_prob(p1_r, y1) - prob_p1_x).item())
                rep_ie_p2[key].append((compute_end_prob(p2_r, y2) - prob_p2_x).item())

        avg_te = np.mean(rep_te) if rep_te else 0.0
        h1_te_list.append(avg_te)
        for key in spec_keys_h1:
            if rep_ie[key]:
                h1_ie[key].append(np.mean(rep_ie[key]))
                h1_ie_p1[key].append(np.mean(rep_ie_p1[key]))
                h1_ie_p2[key].append(np.mean(rep_ie_p2[key]))
        samples_used += 1
        pbar.update(1)
        pbar.set_postfix(TE=f"{np.mean(h1_te_list):.4f}", skipped=batch_idx+1-samples_used)
pbar.close()

# Aggregate
h1_results = {}
for key in spec_keys_h1:
    vs = np.array(h1_ie[key])
    n = len(vs)
    h1_results[key] = {
        "aie_span": float(vs.mean()) if n else 0.0,
        "aie_p1": float(np.mean(h1_ie_p1[key])) if n else 0.0,
        "aie_p2": float(np.mean(h1_ie_p2[key])) if n else 0.0,
        "ci95": float(vs.std() * 1.96 / np.sqrt(n)) if n > 1 else 0.0,
        "n": n,
    }

os.makedirs("experiments/results/H1", exist_ok=True)
with open("experiments/results/H1/h1_results.json", "w") as f:
    json.dump({"results": h1_results, "meta": {"num_samples": samples_used, "avg_te": float(np.mean(h1_te_list))}}, f, indent=2)

print(f"\n{'='*70}")
print("H1 METHOD 2: ROME-STYLE CAUSAL TRACING — RESULTS")
print(f"{'='*70}")
print(f"Samples used: {samples_used} / {NUM_SAMPLES_H1}  |  Noise repeats: {NOISE_REPEATS}  |  σ scale: {NOISE_STD_SCALE}")
print(f"Average Total Effect (TE): {np.mean(h1_te_list):.4f} ± {np.std(h1_te_list)*1.96/np.sqrt(len(h1_te_list)):.4f}")
print(f"TE range: [{np.min(h1_te_list):.4f}, {np.max(h1_te_list):.4f}]  median: {np.median(h1_te_list):.4f}")
print(f"Total sub-layers traced: {len(spec_keys_h1)} (= 7 blocks × 3 passes × 4 components)")

components = ["conv_0", "conv_1", "self_attn", "ffn"]
comp_labels = {"conv_0": "Conv-0", "conv_1": "Conv-1", "self_attn": "Self-Attn", "ffn": "FFN"}

print(f"\n--- Aggregate AIE by Component Type ---")
print(f"{'Component':<14s}  {'Mean AIE':>10s}  {'Std':>8s}  {'Sum AIE':>10s}  {'Count':>6s}")
print("-" * 55)
for c in components:
    vals = [h1_results[k]["aie_span"] for k in h1_results if k.endswith(c)]
    print(f"{comp_labels[c]:<14s}  {np.mean(vals):10.4f}  {np.std(vals):8.4f}  {np.sum(vals):10.4f}  {len(vals):6d}")

conv_sum = sum(h1_results[k]["aie_span"] for k in h1_results if "conv" in k)
attn_sum = sum(h1_results[k]["aie_span"] for k in h1_results if k.endswith("self_attn"))
ffn_sum = sum(h1_results[k]["aie_span"] for k in h1_results if k.endswith("ffn"))
total_sum = conv_sum + attn_sum + ffn_sum
print(f"\nTotal AIE share:  Conv(0+1) = {conv_sum:.4f} ({conv_sum/max(total_sum,1e-8):.1%}),  "
      f"Attn = {attn_sum:.4f} ({attn_sum/max(total_sum,1e-8):.1%}),  FFN = {ffn_sum:.4f} ({ffn_sum/max(total_sum,1e-8):.1%})")

print(f"\n--- Aggregate AIE by Pass ---")
for p in range(3):
    p_vals = [h1_results[k]["aie_span"] for k in h1_results if k.startswith(f"p{p}_")]
    print(f"  Pass {p} → M{p+1}: mean = {np.mean(p_vals):.4f}, sum = {np.sum(p_vals):.4f}, max = {np.max(p_vals):.4f}")

print(f"\n--- Aggregate AIE by Block Depth ---")
for b in range(7):
    b_vals = [h1_results[k]["aie_span"] for k in h1_results if f"_b{b}_" in k]
    print(f"  Block {b}: mean = {np.mean(b_vals):.4f}, sum = {np.sum(b_vals):.4f}")

print(f"\n--- Top 10 Sub-layers by AIE (span) ---")
print(f"{'Rank':<5s} {'Sub-layer':<25s}  {'AIE(span)':>10s}  {'±95%CI':>8s}  {'AIE(p1)':>8s}  {'AIE(p2)':>8s}  {'p1−p2':>8s}  {'N':>4s}")
print("-" * 80)
ranked = sorted(h1_results.items(), key=lambda x: -x[1]["aie_span"])
for rank, (k, r) in enumerate(ranked[:10], 1):
    diff_p = r["aie_p1"] - r["aie_p2"]
    print(f"{rank:<5d} {k:<25s}  {r['aie_span']:10.4f}  {r['ci95']:8.4f}  {r['aie_p1']:8.4f}  {r['aie_p2']:8.4f}  {diff_p:+8.4f}  {r['n']:4d}")

print(f"\n--- Bottom 5 Sub-layers (least causal importance) ---")
for k, r in ranked[-5:]:
    print(f"  {k:<25s}  AIE = {r['aie_span']:.4f} ± {r['ci95']:.4f}")

In [ ]:
## H1 — Norm Diagnostic: Are we measuring importance or output magnitude?
##
## If Conv_1 has much larger output norm than Conv_0, then zero-out ablation
## and causal tracing may simply reflect norm differences, not causal importance.

from experiments.tracer import qanet_forward

N_NORM_SAMPLES = 200
loader_norm = make_loader(dataset, batch_size=1, shuffle=True)

comp_names = ["conv_0", "conv_1", "self_attn", "ffn"]
norm_acc = {}  # key: "p{pass}_b{block}_{comp}" -> list of norms

random.seed(42); torch.manual_seed(42)
samples_norm = 0

with torch.no_grad():
    for Cwid, Ccid, Qwid, Qcid, y1, y2, ids in tqdm(loader_norm, total=N_NORM_SAMPLES, desc="Norm diagnostic"):
        if samples_norm >= N_NORM_SAMPLES:
            break
        Cwid, Ccid = Cwid.to(DEVICE), Ccid.to(DEVICE)
        Qwid, Qcid = Qwid.to(DEVICE), Qcid.to(DEVICE)

        _, _, clean_acts, _ = qanet_forward(model, Cwid, Ccid, Qwid, Qcid, collect=True)

        model_enc_acts = clean_acts.get("model_enc", {})
        for pass_idx in range(3):
            pass_acts = model_enc_acts.get(f"pass_{pass_idx}", {})
            for block_idx in range(7):
                blk_acts = pass_acts.get(f"block_{block_idx}", {})
                for comp in comp_names:
                    act = blk_acts.get(comp)
                    if act is None:
                        continue
                    key = f"p{pass_idx}_b{block_idx}_{comp}"
                    norm_val = act.float().norm().item()
                    norm_acc.setdefault(key, []).append(norm_val)
        samples_norm += 1

print(f"\n{'='*70}")
print("NORM DIAGNOSTIC — Sub-layer Output L2 Norms")
print(f"{'='*70}")
print(f"Samples: {samples_norm}")
print(f"If norm(Conv_1) >> norm(Conv_0), ablation damage reflects magnitude, not importance.\n")

# Aggregate by component type
comp_norms = {c: [] for c in comp_names}
for key, vals in norm_acc.items():
    for c in comp_names:
        if key.endswith(c):
            comp_norms[c].append(np.mean(vals))

print(f"{'Component':<14s}  {'Mean norm':>10s}  {'Std':>10s}  {'Ratio vs Conv_0':>16s}")
print("-" * 55)
conv0_mean = np.mean(comp_norms["conv_0"]) if comp_norms["conv_0"] else 1e-8
for c in comp_names:
    vals = comp_norms[c]
    m = np.mean(vals)
    s = np.std(vals)
    ratio = m / conv0_mean
    print(f"{c:<14s}  {m:10.2f}  {s:10.2f}  {ratio:15.2f}x")

# Aggregate by pass
print(f"\n{'Pass':<10s}  {'Conv_0':>10s}  {'Conv_1':>10s}  {'Attn':>10s}  {'FFN':>10s}")
print("-" * 55)
for p in range(3):
    row = []
    for c in comp_names:
        vals = [np.mean(norm_acc[k]) for k in norm_acc if k.startswith(f"p{p}_") and k.endswith(c)]
        row.append(np.mean(vals) if vals else 0)
    print(f"Pass {p:<5d}  {row[0]:10.2f}  {row[1]:10.2f}  {row[2]:10.2f}  {row[3]:10.2f}")

# Aggregate by block depth
print(f"\n{'Block':<10s}  {'Conv_0':>10s}  {'Conv_1':>10s}  {'Attn':>10s}  {'FFN':>10s}")
print("-" * 55)
for b in range(7):
    row = []
    for c in comp_names:
        vals = [np.mean(norm_acc[k]) for k in norm_acc if f"_b{b}_" in k and k.endswith(c)]
        row.append(np.mean(vals) if vals else 0)
    print(f"Block {b:<5d}  {row[0]:10.2f}  {row[1]:10.2f}  {row[2]:10.2f}  {row[3]:10.2f}")

# The critical test: correlation between norm and AIE
print(f"\n--- Critical Test: Norm vs AIE Correlation ---")
if os.path.exists("experiments/results/H1/h1_results.json"):
    with open("experiments/results/H1/h1_results.json") as f:
        h1_res_check = json.load(f)["results"]

    norm_vals_corr, aie_vals_corr, keys_corr = [], [], []
    for key in h1_res_check:
        if key in norm_acc:
            norm_vals_corr.append(np.mean(norm_acc[key]))
            aie_vals_corr.append(h1_res_check[key]["aie_span"])
            keys_corr.append(key)

    norm_vals_corr = np.array(norm_vals_corr)
    aie_vals_corr = np.array(aie_vals_corr)

    if len(norm_vals_corr) > 2:
        r_norm_aie = np.corrcoef(norm_vals_corr, aie_vals_corr)[0, 1]
        print(f"Pearson r(norm, AIE) = {r_norm_aie:.3f}  (n={len(norm_vals_corr)} sub-layers)")
        if abs(r_norm_aie) > 0.7:
            print("WARNING: Strong correlation — AIE may be a proxy for output magnitude, not causal importance.")
        elif abs(r_norm_aie) > 0.4:
            print("CAUTION: Moderate correlation — norm partially confounds AIE.")
        else:
            print("OK: Weak correlation — AIE is NOT driven by norm. Causal interpretation holds.")

        # ── Norm-Normalized Analysis ──
        aie_per_norm = aie_vals_corr / np.maximum(norm_vals_corr, 1e-8)
        r_after = np.corrcoef(norm_vals_corr, aie_per_norm)[0, 1]

        print(f"\n{'='*70}")
        print("NORM-NORMALIZED AIE (AIE / output_norm)")
        print(f"{'='*70}")
        print(f"After normalization: r(norm, AIE/norm) = {r_after:.3f}")
        if abs(r_after) < 0.3:
            print("Norm confound removed. Residual ranking reflects genuine importance.\n")
        else:
            print(f"Residual correlation still {abs(r_after):.2f}. Partial confound remains.\n")

        # Aggregate normalized AIE by component type
        comp_naie = {c: [] for c in comp_names}
        for i, key in enumerate(keys_corr):
            for c in comp_names:
                if key.endswith(c):
                    comp_naie[c].append(aie_per_norm[i])

        print(f"{'Component':<14s}  {'Raw AIE':>10s}  {'Mean norm':>10s}  {'AIE/norm':>12s}  {'Rank change'}")
        print("-" * 70)
        raw_ranking = sorted(comp_names, key=lambda c: -np.mean([aie_vals_corr[i] for i, k in enumerate(keys_corr) if k.endswith(c)]))
        norm_ranking = sorted(comp_names, key=lambda c: -np.mean(comp_naie[c]) if comp_naie[c] else 0)
        for c in comp_names:
            raw = np.mean([aie_vals_corr[i] for i, k in enumerate(keys_corr) if k.endswith(c)])
            normed = np.mean(comp_naie[c]) if comp_naie[c] else 0
            mn = np.mean([norm_vals_corr[i] for i, k in enumerate(keys_corr) if k.endswith(c)])
            old_rank = raw_ranking.index(c) + 1
            new_rank = norm_ranking.index(c) + 1
            change = "—" if old_rank == new_rank else f"{old_rank}→{new_rank}"
            print(f"{c:<14s}  {raw:10.4f}  {mn:10.1f}  {normed:12.6f}  {change}")

        print(f"\nRaw ranking:       {' > '.join(raw_ranking)}")
        print(f"Normalized ranking: {' > '.join(norm_ranking)}")
        if raw_ranking == norm_ranking:
            print("RANKING PRESERVED after norm correction → importance is real, not just magnitude.")
        else:
            print("RANKING CHANGED → raw AIE was partially driven by norm differences.")

        # Top 10 by normalized AIE
        sorted_idx = np.argsort(-aie_per_norm)
        print(f"\n--- Top 10 Sub-layers by Norm-Normalized AIE ---")
        print(f"{'Rank':<5s} {'Sub-layer':<25s}  {'AIE':>8s}  {'Norm':>8s}  {'AIE/norm':>10s}")
        print("-" * 62)
        for rank, idx in enumerate(sorted_idx[:10], 1):
            print(f"{rank:<5d} {keys_corr[idx]:<25s}  {aie_vals_corr[idx]:8.4f}  {norm_vals_corr[idx]:8.1f}  {aie_per_norm[idx]:10.6f}")

        # Save norm data for later use
        norm_summary = {key: float(np.mean(norm_acc[key])) for key in norm_acc}
        with open("experiments/results/H1/h1_norms.json", "w") as f:
            json.dump({"norms": norm_summary, "r_norm_aie": float(r_norm_aie), "r_after_normalization": float(r_after)}, f, indent=2)
else:
    print("(H1 results not yet saved — run Method 2 first)")

In [ ]:
## [REMOVED] H1 — Content analysis of sub-layer outputs
## Replaced by Experiment 2 (Linear Probe + Discriminativity + Local Coherence)
##   - answer_interior tokens
##   - non-answer tokens
##
## If Conv_1 encodes answer-boundary features, its outputs should
## differ systematically between answer and non-answer positions.

from experiments.tracer import qanet_forward

N_CONTENT = 300
loader_content = make_loader(dataset, batch_size=1, shuffle=True)
random.seed(42); torch.manual_seed(42)

comp_names_c = ["conv_0", "conv_1", "self_attn", "ffn"]
regions = ["answer_start", "answer_end", "answer_interior", "non_answer"]

# Per-token norms by region, aggregated across all (pass, block) positions
token_norms = {c: {r: [] for r in regions} for c in comp_names_c}
# Per-token activation vectors for PCA / direction analysis (only p0_b0)
vecs_by_region = {c: {r: [] for r in regions} for c in comp_names_c}

# Track how much Conv_1 changes the representation vs Conv_0
cos_change = {c: {r: [] for r in regions} for c in ["conv_0", "conv_1"]}

samples_c = 0
with torch.no_grad():
    for Cwid, Ccid, Qwid, Qcid, y1, y2, ids in tqdm(loader_content, total=N_CONTENT, desc="Content analysis"):
        if samples_c >= N_CONTENT:
            break
        Cwid, Ccid = Cwid.to(DEVICE), Ccid.to(DEVICE)
        Qwid, Qcid = Qwid.to(DEVICE), Qcid.to(DEVICE)
        y1_val, y2_val = y1.item(), y2.item()

        if y1_val >= Cwid.size(1) or y2_val >= Cwid.size(1) or y1_val > y2_val:
            continue

        _, _, clean_acts, _ = qanet_forward(model, Cwid, Ccid, Qwid, Qcid, collect=True)

        L = Cwid.size(1)
        region_map = np.full(L, 3, dtype=int)  # 3 = non_answer
        region_map[y1_val] = 0                   # answer_start
        region_map[y2_val] = 1                   # answer_end
        if y2_val > y1_val + 1:
            region_map[y1_val+1:y2_val] = 2      # answer_interior

        model_enc_acts = clean_acts.get("model_enc", {})
        for pass_idx in range(3):
            pass_acts = model_enc_acts.get(f"pass_{pass_idx}", {})
            for block_idx in range(7):
                blk_acts = pass_acts.get(f"block_{block_idx}", {})
                for comp in comp_names_c:
                    act = blk_acts.get(comp)  # [1, d_model, L]
                    if act is None:
                        continue
                    act_np = act[0].cpu().float().numpy()  # [d_model, L]
                    per_token_norm = np.linalg.norm(act_np, axis=0)  # [L]

                    for t in range(min(L, len(region_map))):
                        r_idx = region_map[t]
                        token_norms[comp][regions[r_idx]].append(per_token_norm[t])

                    # Collect vectors from p0_b0 for deeper analysis
                    if pass_idx == 0 and block_idx == 0:
                        for r_idx, r_name in enumerate(regions):
                            mask = (region_map == r_idx)
                            if mask.any():
                                vecs = act_np[:, mask].T  # [n_tokens, d_model]
                                vecs_by_region[comp][r_name].append(vecs)

        # Representational change: cosine(input_to_conv, output_of_conv)
        # For p0_b0: input to conv_0 is the position-encoded input
        # conv_0 output is in blk_acts["conv_0"], conv_1 in blk_acts["conv_1"]
        blk0 = model_enc_acts.get("pass_0", {}).get("block_0", {})
        for cname in ["conv_0", "conv_1"]:
            act = blk0.get(cname)
            if act is None:
                continue
            a = act[0].cpu().float().numpy()  # [d_model, L]
            # Get the "input" to this conv (the block output of previous step)
            if cname == "conv_0":
                # input is position-encoded M0 — approximated by block input
                inp = clean_acts.get("cq_resized")
                if inp is not None:
                    inp = inp[0].cpu().float().numpy()
                else:
                    continue
            else:
                # input to conv_1 = residual + conv_0 output ≈ previous stage output
                # We don't have it exactly, so skip this comparison
                continue

            for t in range(min(a.shape[1], inp.shape[1], len(region_map))):
                r_idx = region_map[t]
                a_t = a[:, t]
                inp_t = inp[:, t]
                cos = np.dot(a_t, inp_t) / (np.linalg.norm(a_t) * np.linalg.norm(inp_t) + 1e-10)
                cos_change[cname][regions[r_idx]].append(cos)

        samples_c += 1

print(f"\n{'='*70}")
print("CONV_1 CONTENT ANALYSIS — What information does each sub-layer encode?")
print(f"{'='*70}")
print(f"Samples: {samples_c}")

print(f"\n--- Per-token Output Norm by Position Type ---")
print(f"{'Component':<14s}  {'ans_start':>10s}  {'ans_end':>10s}  {'ans_interior':>12s}  {'non_answer':>11s}  {'start/non':>10s}")
print("-" * 75)
for c in comp_names_c:
    vals = {r: np.mean(token_norms[c][r]) if token_norms[c][r] else 0 for r in regions}
    ratio = vals["answer_start"] / max(vals["non_answer"], 1e-8)
    print(f"{c:<14s}  {vals['answer_start']:10.2f}  {vals['answer_end']:10.2f}  "
          f"{vals['answer_interior']:12.2f}  {vals['non_answer']:11.2f}  {ratio:10.3f}x")

print(f"\n--- Key Question: Does Conv_1 produce higher activation at answer boundaries? ---")
for c in comp_names_c:
    start_norm = np.mean(token_norms[c]["answer_start"]) if token_norms[c]["answer_start"] else 0
    non_norm = np.mean(token_norms[c]["non_answer"]) if token_norms[c]["non_answer"] else 0
    diff_pct = (start_norm - non_norm) / max(non_norm, 1e-8) * 100
    print(f"  {c:<14s}: answer_start norm = {start_norm:.2f}, non_answer norm = {non_norm:.2f}, "
          f"diff = {diff_pct:+.1f}%")

print(f"\n--- Variance of per-token norms (how selective is each component?) ---")
print(f"{'Component':<14s}  {'Std(norm)':>10s}  {'CV (std/mean)':>14s}")
print("-" * 42)
for c in comp_names_c:
    all_norms = []
    for r in regions:
        all_norms.extend(token_norms[c][r])
    if all_norms:
        m, s = np.mean(all_norms), np.std(all_norms)
        print(f"{c:<14s}  {s:10.2f}  {s/max(m,1e-8):14.3f}")

# Cosine change analysis
if any(cos_change["conv_0"][r] for r in regions):
    print(f"\n--- Conv_0 output direction vs its input (p0_b0) ---")
    print(f"  High cosine = mostly preserves input direction. Low cosine = adds orthogonal info.")
    for r in regions:
        vals = cos_change["conv_0"][r]
        if vals:
            print(f"  {r:>16s}: cos = {np.mean(vals):.4f} ± {np.std(vals):.4f}")

In [ ]:
## [REMOVED] H1 — Causal Tracing Heatmaps (auxiliary visualization, not referenced in report)
print("  1. AIE heatmap (4 component types × 21 block-pass positions)")
print("  2. Aggregate AIE bar chart by component type")
print("  3. Start vs End position bias heatmap (AIE_p1 − AIE_p2)\n")

components = ["conv_0", "conv_1", "self_attn", "ffn"]
n_passes, n_blocks = 3, 7

# --- 1. Main heatmap ---
matrix = np.zeros((4, 21))
for p in range(3):
    for b in range(7):
        col = p * 7 + b
        for ci, comp in enumerate(components):
            matrix[ci, col] = h1_results.get(f"p{p}_b{b}_{comp}", {}).get("aie_span", 0)

fig, ax = plt.subplots(figsize=(18, 4))
vmax = max(abs(matrix.max()), abs(matrix.min()), 1e-6)
im = ax.imshow(matrix, cmap="RdYlBu_r", aspect="auto", vmin=-vmax*0.1, vmax=vmax)
ax.set_yticks(range(4)); ax.set_yticklabels(["Conv 0", "Conv 1", "Self-Attn", "FFN"])
ax.set_xticks(range(21)); ax.set_xticklabels([f"B{b}" for _ in range(3) for b in range(7)], fontsize=7)
for p in range(1, 3):
    ax.axvline(x=p*7-0.5, color="white", lw=2)
for p in range(3):
    ax.text(p*7+3, -0.8, f"Pass {p+1}→M{p+1}", ha="center", fontsize=10, fontweight="bold")
plt.colorbar(im, ax=ax, label="AIE (span)", shrink=0.8)
ax.set_title("H1: Component-Level Causal Tracing — QANet Model Encoder")
fig.tight_layout(); plt.show()

# --- 2. Aggregate bars ---
agg = {c: [] for c in components}
for key, val in h1_results.items():
    for c in components:
        if key.endswith(c):
            agg[c].append(val["aie_span"])
means = [np.mean(agg[c]) for c in components]
errs = [np.std(agg[c])/np.sqrt(len(agg[c]))*1.96 if agg[c] else 0 for c in components]

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(["Conv 0", "Conv 1", "Self-Attn", "FFN"], means, yerr=errs,
       color=["#2196F3","#42A5F5","#FF9800","#4CAF50"], capsize=5, edgecolor="black")
ax.set_ylabel("Average IE (span)"); ax.set_title("H1: Aggregate Causal Effect by Component")
ax.axhline(0, color="gray", ls="--", alpha=0.5); fig.tight_layout(); plt.show()

# --- 3. Start vs End difference ---
diff = np.zeros((4, 21))
for p in range(3):
    for b in range(7):
        col = p*7+b
        for ci, comp in enumerate(components):
            v = h1_results.get(f"p{p}_b{b}_{comp}", {})
            diff[ci, col] = v.get("aie_p1", 0) - v.get("aie_p2", 0)
fig, ax = plt.subplots(figsize=(18, 4))
vmax = max(abs(diff.max()), abs(diff.min()), 1e-6)
im = ax.imshow(diff, cmap="RdBu", aspect="auto", vmin=-vmax, vmax=vmax)
ax.set_yticks(range(4)); ax.set_yticklabels(["Conv 0","Conv 1","Self-Attn","FFN"])
ax.set_xticks(range(21)); ax.set_xticklabels([f"B{b}" for _ in range(3) for b in range(7)], fontsize=7)
for p in range(1,3): ax.axvline(x=p*7-0.5, color="black", lw=2)
plt.colorbar(im, ax=ax, label="AIE_p1 − AIE_p2  (red→start, blue→end)")
ax.set_title("H1: Start vs End Position Bias"); fig.tight_layout(); plt.show()

In [ ]:
## H1 — Intervention Spectrum: Zero / Noise / Mean Replacement
##
## Four conditions to separate information content from magnitude:
##   original  — no intervention (baseline)
##   mean      — replace with dataset-average activation (in-distribution, removes sample-specific info)
##   noise     — replace with norm-matched random noise (out-of-distribution, removes all info)
##   zero      — replace with zeros (removes info + norm)
##
## Expected ordering if SAMPLE-SPECIFIC INFORMATION drives importance:
##   damage: zero ≈ noise > mean >> original
## Expected ordering if only NORM drives importance:
##   damage: zero >> noise ≈ mean (noise/mean preserve norm, so less shift)
##
## LIMITATIONS:
##   1. Mean computed from 500 samples; approximates population mean.
##   2. Sequence lengths vary; mean activation is truncated to match each sample.
##   3. Mean replacement is in-distribution on average but each specific sample sees a
##      mis-matched activation — this still introduces some distributional shift,
##      so the "mean damage" is an UPPER BOUND on the true fixed-contribution damage.
##   4. All interventions are global (all 21 blocks simultaneously), so results reflect
##      collective rather than per-block contribution.

import importlib, experiments.tracer, random
importlib.reload(experiments.tracer)
from experiments.tracer import qanet_forward, SkipSpec
from EvaluateTools.eval_utils import convert_tokens, squad_evaluate
from tqdm.auto import tqdm

print("=" * 70)
print("H1: INTERVENTION SPECTRUM — Zero / Noise / Mean Replacement")
print("=" * 70)

# ── Step 1: Collect mean activations across 500 samples ──
print("Step 1: Collecting mean activations from 500 samples...\n")
N_MEAN = 500
loader_mean = make_loader(dataset, batch_size=1, shuffle=True)
random.seed(42); torch.manual_seed(42)
mean_sums = {}
mean_counts = {}
comp_list = ["conv_0", "conv_1", "self_attn", "ffn"]

samples_mean = 0
with torch.no_grad():
    for Cwid, Ccid, Qwid, Qcid, y1, y2, ids in tqdm(loader_mean, total=N_MEAN, desc="Collecting means"):
        if samples_mean >= N_MEAN:
            break
        Cwid, Ccid = Cwid.to(DEVICE), Ccid.to(DEVICE)
        Qwid, Qcid = Qwid.to(DEVICE), Qcid.to(DEVICE)
        _, _, acts, _ = qanet_forward(model, Cwid, Ccid, Qwid, Qcid, collect=True)

        model_enc = acts.get("model_enc", {})
        for pi in range(3):
            for bi in range(7):
                blk = model_enc.get(f"pass_{pi}", {}).get(f"block_{bi}", {})
                for c in comp_list:
                    act = blk.get(c)
                    if act is None:
                        continue
                    key = (pi, bi, c)
                    if key not in mean_sums:
                        mean_sums[key] = torch.zeros_like(act[0])
                        mean_counts[key] = 0
                    L = min(mean_sums[key].shape[-1], act.shape[-1])
                    mean_sums[key][..., :L] += act[0, ..., :L].float()
                    mean_counts[key] += 1
        samples_mean += 1

mean_acts = {}
for key, s in mean_sums.items():
    mean_acts[key] = (s / mean_counts[key]).to(DEVICE)

print(f"Collected means from {samples_mean} samples for {len(mean_acts)} (pass, block, comp) positions.\n")

# ── Step 2: Subsampled baseline ──
print("Step 2: Computing subsampled baseline...")
MAX_BATCHES_NR = 64
loader_base_nr = make_loader(dataset, batch_size=32, shuffle=False)
answers_base_nr = {}
with torch.no_grad():
    for bi, (Cwid, Ccid, Qwid, Qcid, y1, y2, ids) in enumerate(tqdm(loader_base_nr, desc="  baseline", total=MAX_BATCHES_NR)):
        if bi >= MAX_BATCHES_NR:
            break
        Cwid, Ccid = Cwid.to(DEVICE), Ccid.to(DEVICE)
        Qwid, Qcid = Qwid.to(DEVICE), Qcid.to(DEVICE)
        p1, p2, _, _ = qanet_forward(model, Cwid, Ccid, Qwid, Qcid, skip_spec=None)
        p1_log = F.log_softmax(p1, dim=1)
        p2_log = F.log_softmax(p2, dim=1)
        outer = p1_log.unsqueeze(2) + p2_log.unsqueeze(1)
        tri = torch.triu(torch.ones(outer.size(-2), outer.size(-1), device=DEVICE, dtype=torch.bool))
        outer = outer.masked_fill(~tri, float('-inf'))
        flat = outer.view(outer.size(0), -1)
        idx = torch.argmax(flat, dim=1)
        L = p1.size(1)
        ymin = idx // L; ymax = idx % L
        ans, _ = convert_tokens(eval_file, ids.tolist(), ymin.tolist(), ymax.tolist())
        answers_base_nr.update(ans)
nr_base = squad_evaluate(eval_file, answers_base_nr)
nr_base_f1 = nr_base["f1"]
nr_base_em = nr_base["exact_match"]
print(f"Subsampled baseline ({len(answers_base_nr)} samples): F1={nr_base_f1:.2f}, EM={nr_base_em:.2f}\n")

# ── Step 3: Run all intervention conditions ──
print("Step 3: Running 12 intervention conditions (4 components × 3 modes)...\n")
INTERV_CONFIGS = {}
for comp, skip_key in [("conv_1", "conv_1"), ("conv_0", "conv_0"), ("attn", "self_attn"), ("ffn", "ffn")]:
    INTERV_CONFIGS[f"zero_{comp}"]  = SkipSpec(global_skip={skip_key}, mode="zero")
    INTERV_CONFIGS[f"noise_{comp}"] = SkipSpec(global_skip={skip_key}, mode="noise")
    INTERV_CONFIGS[f"mean_{comp}"]  = SkipSpec(global_skip={skip_key}, mode="mean", mean_acts=mean_acts)

interv_results = {}
for cfg_name, skip in tqdm(INTERV_CONFIGS.items(), desc="Interventions"):
    loader_iv = make_loader(dataset, batch_size=32, shuffle=False)
    answers = {}
    with torch.no_grad():
        for bi, (Cwid, Ccid, Qwid, Qcid, y1, y2, ids) in enumerate(loader_iv):
            if bi >= MAX_BATCHES_NR:
                break
            Cwid, Ccid = Cwid.to(DEVICE), Ccid.to(DEVICE)
            Qwid, Qcid = Qwid.to(DEVICE), Qcid.to(DEVICE)
            p1, p2, _, _ = qanet_forward(model, Cwid, Ccid, Qwid, Qcid, skip_spec=skip)
            p1_log = F.log_softmax(p1, dim=1)
            p2_log = F.log_softmax(p2, dim=1)
            outer = p1_log.unsqueeze(2) + p2_log.unsqueeze(1)
            tri = torch.triu(torch.ones(outer.size(-2), outer.size(-1), device=DEVICE, dtype=torch.bool))
            outer = outer.masked_fill(~tri, float('-inf'))
            flat = outer.view(outer.size(0), -1)
            idx = torch.argmax(flat, dim=1)
            L = p1.size(1)
            ymin = idx // L; ymax = idx % L
            ans, _ = convert_tokens(eval_file, ids.tolist(), ymin.tolist(), ymax.tolist())
            answers.update(ans)
    metrics = squad_evaluate(eval_file, answers)
    interv_results[cfg_name] = {"f1": metrics["f1"], "em": metrics["exact_match"]}

os.makedirs("experiments/results/H1", exist_ok=True)
with open("experiments/results/H1/h1_intervention_spectrum.json", "w") as f:
    json.dump(interv_results, f, indent=2)

# ── Results Table ──
print(f"\n{'='*70}")
print("INTERVENTION SPECTRUM RESULTS")
print(f"{'='*70}")
print(f"\n{'':30s}  {'Information preserved →':^50s}")
print(f"{'Component':<10s}  {'Zero':>10s}  {'Noise':>10s}  {'Mean':>10s}  {'Baseline':>10s}")
print(f"{'':10s}  {'(none)':>10s}  {'(norm only)':>10s}  {'(avg struct)':>10s}  {'(all)':>10s}")
print("-" * 55)
for comp in ["conv_1", "conv_0", "attn", "ffn"]:
    z = interv_results[f"zero_{comp}"]["f1"]
    n = interv_results[f"noise_{comp}"]["f1"]
    m = interv_results[f"mean_{comp}"]["f1"]
    print(f"{comp:<10s}  {z:10.2f}  {n:10.2f}  {m:10.2f}  {nr_base_f1:10.2f}")

print(f"\n{'Component':<10s}  {'Zero ΔF1':>10s}  {'Noise ΔF1':>10s}  {'Mean ΔF1':>10s}")
print("-" * 45)
for comp, label in [("conv_1", "Conv_1"), ("conv_0", "Conv_0"), ("attn", "Attn"), ("ffn", "FFN")]:
    z_d = interv_results[f"zero_{comp}"]["f1"] - nr_base_f1
    n_d = interv_results[f"noise_{comp}"]["f1"] - nr_base_f1
    m_d = interv_results[f"mean_{comp}"]["f1"] - nr_base_f1
    print(f"{label:<10s}  {z_d:+10.2f}  {n_d:+10.2f}  {m_d:+10.2f}")

# ── Interpretation ──
print(f"\n--- Causal Decomposition ---")
print("Logic: baseline → mean = removes sample-specific info (keeps avg structure)")
print("       baseline → noise = removes all structured info (keeps norm)")
print("       baseline → zero  = removes everything\n")
for comp, label in [("conv_1", "Conv_1"), ("conv_0", "Conv_0"), ("attn", "Attn"), ("ffn", "FFN")]:
    z_drop = nr_base_f1 - interv_results[f"zero_{comp}"]["f1"]
    n_drop = nr_base_f1 - interv_results[f"noise_{comp}"]["f1"]
    m_drop = nr_base_f1 - interv_results[f"mean_{comp}"]["f1"]
    sample_specific = m_drop
    structural = z_drop - n_drop if z_drop > n_drop else 0
    print(f"  {label}:")
    print(f"    Total damage (zero):   {z_drop:6.2f} F1")
    print(f"    Noise damage:          {n_drop:6.2f} F1  (OOD interference + info loss)")
    print(f"    Mean damage:           {m_drop:6.2f} F1  ≈ sample-specific info loss")
    if m_drop > 1.0:
        print(f"    → {label} MUST produce INPUT-SPECIFIC features (mean replacement still hurts {m_drop:.1f} F1)")
    elif m_drop > 0.1:
        print(f"    → {label} has MODERATE sample-specificity (mean recovers most but not all)")
    else:
        print(f"    → {label} mostly provides a FIXED contribution (mean replacement is sufficient)")
    print()

print(f"--- Key Comparison: Noise vs Mean ---")
print("If noise damage >> mean damage → noise experiment was confounded by OOD interference.")
print("If noise damage ≈ mean damage → noise result was trustworthy.\n")
for comp, label in [("conv_1", "Conv_1"), ("conv_0", "Conv_0"), ("attn", "Attn"), ("ffn", "FFN")]:
    n_drop = nr_base_f1 - interv_results[f"noise_{comp}"]["f1"]
    m_drop = nr_base_f1 - interv_results[f"mean_{comp}"]["f1"]
    diff = n_drop - m_drop
    print(f"  {label}: noise_drop={n_drop:.2f}, mean_drop={m_drop:.2f}, gap={diff:+.2f}")
    if diff > 2.0:
        print(f"         → Large gap: noise result had significant OOD confound")
    elif diff > 0.5:
        print(f"         → Moderate gap: some OOD effect in noise condition")
    else:
        print(f"         → Small gap: noise and mean agree well")

print(f"\n--- Limitations ---")
print("  1. Mean is computed from 500 samples; approximates but may not perfectly represent population mean.")
print("  2. Sequence lengths vary; mean activation is truncated to match each sample's length.")
print("  3. Mean replacement is in-distribution *on average* but each specific sample sees a")
print("     mismatched activation — so 'mean damage' is an UPPER BOUND on true fixed-contribution damage.")
print("  4. All interventions are global (all 21 blocks simultaneously), capturing collective importance.")

In [ ]:
## H1 — Linear Probe: What information does Conv_1 encode?
##
## Train a linear classifier on each sub-layer's per-token output:
##   label = 1 if token is inside the answer span, 0 otherwise.
## Compare probe accuracy across Conv_0 / Conv_1 / Self-Attn / FFN.
## If Conv_1 probe >> Conv_0 probe → Conv_1 encodes answer-relevant features
## that Conv_0 (same architecture) does not.

import importlib, experiments.tracer
importlib.reload(experiments.tracer)
from experiments.tracer import qanet_forward
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

N_PROBE = 500
PROBE_BLOCKS = [(0, 0), (0, 3), (0, 6), (1, 0), (2, 0)]  # sample of (pass, block)
loader_probe = make_loader(dataset, batch_size=1, shuffle=True)
random.seed(42); torch.manual_seed(42)

comp_names_probe = ["conv_0", "conv_1", "self_attn", "ffn"]

# Collect features: {(pass, block, comp): {"X": [...], "y": [...]}}
probe_data = {}
for pb in PROBE_BLOCKS:
    for c in comp_names_probe:
        probe_data[(pb[0], pb[1], c)] = {"X": [], "y": []}

samples_probe = 0
with torch.no_grad():
    for Cwid, Ccid, Qwid, Qcid, y1, y2, ids in tqdm(loader_probe, total=N_PROBE, desc="Probe features"):
        if samples_probe >= N_PROBE:
            break
        y1_val, y2_val = y1.item(), y2.item()
        L = Cwid.size(1)
        if y1_val >= L or y2_val >= L or y1_val > y2_val:
            continue

        Cwid, Ccid = Cwid.to(DEVICE), Ccid.to(DEVICE)
        Qwid, Qcid = Qwid.to(DEVICE), Qcid.to(DEVICE)

        _, _, clean_acts, _ = qanet_forward(model, Cwid, Ccid, Qwid, Qcid, collect=True)

        # Labels: 1 for answer tokens, 0 for non-answer (only non-padding)
        pad_mask = (Cwid[0].cpu() != 0).numpy()  # True for real tokens
        labels = np.zeros(L, dtype=int)
        labels[y1_val:y2_val+1] = 1

        model_enc_acts = clean_acts.get("model_enc", {})
        for (pi, bi) in PROBE_BLOCKS:
            blk_acts = model_enc_acts.get(f"pass_{pi}", {}).get(f"block_{bi}", {})
            for c in comp_names_probe:
                act = blk_acts.get(c)
                if act is None:
                    continue
                feat = act[0].cpu().float().numpy()  # [d_model, L]
                for t in range(L):
                    if not pad_mask[t]:
                        continue
                    probe_data[(pi, bi, c)]["X"].append(feat[:, t])
                    probe_data[(pi, bi, c)]["y"].append(labels[t])

        samples_probe += 1

# Train probes
print(f"\n{'='*70}")
print("LINEAR PROBE — Can a linear classifier decode answer positions?")
print(f"{'='*70}")
print(f"Samples: {samples_probe}, probe = LogisticRegression on per-token vectors")
print(f"Label: 1 = answer token, 0 = non-answer token")
print(f"Train on 80%, test on 20%. Metric: ROC-AUC (handles class imbalance).\n")

probe_results = {}
print(f"{'(pass,block)':<14s}  {'Component':<12s}  {'AUC':>6s}  {'Acc':>6s}  {'F1(ans)':>8s}  {'N_train':>8s}  {'%pos':>6s}")
print("-" * 70)

for (pi, bi) in PROBE_BLOCKS:
    for c in comp_names_probe:
        key = (pi, bi, c)
        X = np.array(probe_data[key]["X"])
        y = np.array(probe_data[key]["y"])
        if len(X) < 100 or y.sum() < 10:
            continue

        # Train/test split (deterministic)
        n = len(X)
        idx = np.arange(n)
        np.random.seed(42)
        np.random.shuffle(idx)
        split = int(0.8 * n)
        X_tr, X_te = X[idx[:split]], X[idx[split:]]
        y_tr, y_te = y[idx[:split]], y[idx[split:]]

        clf = LogisticRegression(max_iter=1000, class_weight="balanced", C=1.0)
        clf.fit(X_tr, y_tr)
        y_pred = clf.predict(X_te)
        y_prob = clf.predict_proba(X_te)[:, 1]

        auc = roc_auc_score(y_te, y_prob)
        acc = accuracy_score(y_te, y_pred)
        f1 = f1_score(y_te, y_pred)
        pct_pos = y.mean() * 100

        probe_results[key] = {"auc": auc, "acc": acc, "f1_ans": f1, "n": len(X), "pct_pos": pct_pos}
        print(f"p{pi}_b{bi}        {c:<12s}  {auc:6.3f}  {acc:6.3f}  {f1:8.3f}  {len(X):8d}  {pct_pos:5.1f}%")
    print()

# Summary: average AUC by component type
print(f"--- Average AUC by Component Type (across all probed blocks) ---")
for c in comp_names_probe:
    aucs = [v["auc"] for k, v in probe_results.items() if k[2] == c]
    if aucs:
        print(f"  {c:<14s}: mean AUC = {np.mean(aucs):.3f}  (n={len(aucs)} blocks)")

print(f"\n--- Average AUC by Block Location ---")
for (pi, bi) in PROBE_BLOCKS:
    aucs = [v["auc"] for k, v in probe_results.items() if k[0] == pi and k[1] == bi]
    if aucs:
        print(f"  p{pi}_b{bi}: mean AUC = {np.mean(aucs):.3f}")

# The critical comparison
conv1_aucs = [v["auc"] for k, v in probe_results.items() if k[2] == "conv_1"]
conv0_aucs = [v["auc"] for k, v in probe_results.items() if k[2] == "conv_0"]
attn_aucs = [v["auc"] for k, v in probe_results.items() if k[2] == "self_attn"]
if conv1_aucs and conv0_aucs:
    print(f"\n--- Conv_1 vs Conv_0: Does Conv_1 encode more answer information? ---")
    print(f"  Conv_1 mean AUC: {np.mean(conv1_aucs):.3f}")
    print(f"  Conv_0 mean AUC: {np.mean(conv0_aucs):.3f}")
    print(f"  Attn   mean AUC: {np.mean(attn_aucs):.3f}")
    diff = np.mean(conv1_aucs) - np.mean(conv0_aucs)
    if diff > 0.02:
        print(f"  Conv_1 encodes MORE answer-position information than Conv_0 (+{diff:.3f} AUC)")
    elif diff < -0.02:
        print(f"  Conv_0 encodes MORE answer-position information than Conv_1 ({diff:+.3f} AUC)")
    else:
        print(f"  Similar answer-position encoding (diff = {diff:+.3f} AUC)")

os.makedirs("experiments/results/H1", exist_ok=True)
probe_save = {f"p{k[0]}_b{k[1]}_{k[2]}": v for k, v in probe_results.items()}
with open("experiments/results/H1/h1_linear_probe.json", "w") as f:
    json.dump(probe_save, f, indent=2)

In [ ]:
## H1 — Experiment 2c: Local Coherence (Spatial Autocorrelation)
##
## Measures mean cosine similarity between output vectors at positions t and t+lag.
## Quantifies the spatial correlation structure each component imposes.
## Corresponds to report Experiment 2c.

import importlib, experiments.tracer
importlib.reload(experiments.tracer)
from experiments.tracer import qanet_forward
import numpy as np

N_COH = 500
comp_names_coh = ["conv_0", "conv_1", "self_attn", "ffn"]
coherence = {c: {lag: [] for lag in [1, 2, 3, 5]} for c in comp_names_coh}

loader_coh = make_loader(dataset, batch_size=1, shuffle=True)
random.seed(42); torch.manual_seed(42)

n_coh = 0
with torch.no_grad():
    for Cwid, Ccid, Qwid, Qcid, y1, y2, ids in tqdm(loader_coh, total=N_COH, desc="Exp2c: Local Coherence"):
        if n_coh >= N_COH:
            break
        L = Cwid.size(1)
        if y1.item() >= L or y2.item() >= L:
            continue

        Cwid_d, Ccid_d = Cwid.to(DEVICE), Ccid.to(DEVICE)
        Qwid_d, Qcid_d = Qwid.to(DEVICE), Qcid.to(DEVICE)
        _, _, clean_acts, _ = qanet_forward(model, Cwid_d, Ccid_d, Qwid_d, Qcid_d, collect=True)

        pad_mask = (Cwid[0].cpu() != 0).numpy()
        me = clean_acts.get("model_enc", {})
        ba = me.get("pass_0", {}).get("block_0", {})

        for c in comp_names_coh:
            act = ba.get(c)
            if act is None:
                continue
            feat = act[0].cpu().float().numpy()
            nrm = np.linalg.norm(feat, axis=0)
            ok = pad_mask & (nrm > 1e-8)
            for lag in [1, 2, 3, 5]:
                for t in range(L - lag):
                    if ok[t] and ok[t + lag]:
                        cs = np.dot(feat[:, t], feat[:, t + lag]) / (nrm[t] * nrm[t + lag])
                        coherence[c][lag].append(cs)
        n_coh += 1

print(f"\n{'='*80}")
print(f"H1 Experiment 2c: Local Coherence — Spatial Autocorrelation (p0_b0)")
print(f"{'='*80}")
print(f"Samples: {n_coh}")
print(f"Metric: mean cosine similarity between output vectors at positions t and t+lag\n")

print(f"{'Component':<14s}  {'lag=1':>8s}  {'lag=2':>8s}  {'lag=3':>8s}  {'lag=5':>8s}  {'decay(1→5)':>11s}")
print("-" * 60)
coh_summary = {}
for c in comp_names_coh:
    v = {}
    for lag in [1, 2, 3, 5]:
        v[lag] = np.mean(coherence[c][lag]) if coherence[c][lag] else 0
    decay = v[1] - v[5]
    coh_summary[c] = v
    print(f"{c:<14s}  {v[1]:8.4f}  {v[2]:8.4f}  {v[3]:8.4f}  {v[5]:8.4f}  {decay:11.4f}")

print(f"\nInterpretation:")
print(f"  Higher lag-1 = more locally coherent output")
print(f"  Larger decay(1->5) = features are spatially structured (local, not global)")

os.makedirs("experiments/results/H1", exist_ok=True)
coh_save = {c: {str(lag): float(np.mean(vs)) if vs else 0.0 for lag, vs in d.items()} for c, d in coherence.items()}
with open("experiments/results/H1/h1_local_coherence.json", "w") as f:
    json.dump(coh_save, f, indent=2)
print(f"Results saved to experiments/results/H1/h1_local_coherence.json")

In [ ]:
## H1 — Attention Pattern Degradation: Positive evidence for Conv_1's role
##
## Core idea: If Conv_1 is the "local feature substrate" for Self-Attention,
## removing Conv_1 should degrade attention patterns (higher entropy, lost focus).
##
## Conditions:  clean  |  -Conv_1 (critical)  |  -Conv_0 (control)
## Metrics per sample, per block:
##   1. Attention entropy  (higher = more uniform = degraded)
##   2. JS divergence vs clean  (how much the pattern shifted)
##   3. Attention mass on answer-span tokens  (focus lost?)
##
## Memory-efficient: compute metrics on-the-fly per sample; only keep stats.

import importlib, experiments.tracer
importlib.reload(experiments.tracer)
from experiments.tracer import qanet_forward, SkipSpec
from Models.encoder import mask_logits

print("=" * 70)
print("H1: ATTENTION PATTERN DEGRADATION — Does Conv_1 feed Self-Attention?")
print("=" * 70)

# ── Metric helpers (operate on single attention tensors) ──
def _entropy(attn):
    """attn: [1, H, L, L] → scalar mean entropy."""
    eps = 1e-10
    return -(attn * (attn + eps).log()).sum(dim=-1).mean().item()

def _js_div(p, q):
    """JS divergence between two [1, H, L, L] attention tensors."""
    eps = 1e-10
    m = 0.5 * (p + q)
    kl_pm = (p * ((p + eps) / (m + eps)).log()).sum(dim=-1).mean()
    kl_qm = (q * ((q + eps) / (m + eps)).log()).sum(dim=-1).mean()
    return (0.5 * (kl_pm + kl_qm)).item()

def _ans_mass(attn, s, e):
    """Fraction of attention on answer span [s, e]. attn: [1, H, L, L]."""
    if s > e or e >= attn.shape[-1]:
        return float('nan')
    return attn[:, :, :, s:e+1].sum(dim=-1).mean().item()

# ── Hook: recompute attention weights from Q,K inside self_att ──
_attn_store = {}

def _make_hook(name):
    def hook(module, inputs, output):
        x, mask = inputs[0], inputs[1]
        sa = module
        B, C, L = x.size()
        x_t = x.transpose(1, 2)
        q = sa.q_linear(x_t).view(B, L, sa.num_heads, sa.d_k)
        k = sa.k_linear(x_t).view(B, L, sa.num_heads, sa.d_k)
        q = q.permute(2, 0, 1, 3).contiguous().view(B * sa.num_heads, L, sa.d_k)
        k = k.permute(2, 0, 1, 3).contiguous().view(B * sa.num_heads, L, sa.d_k)
        m = mask.bool() if mask.dtype != torch.bool else mask
        am = m.unsqueeze(1).expand(-1, L, -1).repeat(sa.num_heads, 1, 1)
        scores = torch.bmm(q, k.transpose(1, 2)) * sa.scale
        scores = mask_logits(scores, am)
        attn = F.softmax(scores, dim=2)
        _attn_store[name] = attn.view(sa.num_heads, B, L, L).permute(1, 0, 2, 3).detach()
    return hook

handles = []
for bi, blk in enumerate(model.model_enc_blks):
    handles.append(blk.self_att.register_forward_hook(_make_hook(f"b{bi}")))
print(f"Registered hooks on {len(handles)} encoder blocks.\n")

# ── Main loop: per-sample, compute metrics on-the-fly ──
N_SAMPLES = 50
CONDITIONS = {
    "clean":       None,
    "skip_conv_1": SkipSpec(global_skip={"conv_1"}, mode="zero"),
    "skip_conv_0": SkipSpec(global_skip={"conv_0"}, mode="zero"),
}
COND_NAMES = list(CONDITIONS.keys())

ent_acc  = {c: [[] for _ in range(7)] for c in COND_NAMES}
jsd_acc  = {c: [[] for _ in range(7)] for c in ["skip_conv_1", "skip_conv_0"]}
am_acc   = {c: [[] for _ in range(7)] for c in COND_NAMES}

loader_ad = make_loader(dataset, batch_size=1, shuffle=True)
random.seed(123); torch.manual_seed(123)

sample_count = 0
with torch.no_grad():
    for Cwid, Ccid, Qwid, Qcid, y1, y2, ids in tqdm(loader_ad, total=N_SAMPLES, desc="Attn degradation"):
        if sample_count >= N_SAMPLES:
            break
        Cwid, Ccid = Cwid.to(DEVICE), Ccid.to(DEVICE)
        Qwid, Qcid = Qwid.to(DEVICE), Qcid.to(DEVICE)
        ans_s, ans_e = y1.item(), y2.item()

        sample_attn = {}
        for cond_name, skip in CONDITIONS.items():
            _attn_store.clear()
            qanet_forward(model, Cwid, Ccid, Qwid, Qcid, skip_spec=skip)
            sample_attn[cond_name] = {f"b{bi}": _attn_store[f"b{bi}"].cpu() for bi in range(7) if f"b{bi}" in _attn_store}

        for bi in range(7):
            key = f"b{bi}"
            a_clean = sample_attn["clean"].get(key)
            a_c1    = sample_attn["skip_conv_1"].get(key)
            a_c0    = sample_attn["skip_conv_0"].get(key)
            if a_clean is None:
                continue

            ent_acc["clean"][bi].append(_entropy(a_clean))
            ent_acc["skip_conv_1"][bi].append(_entropy(a_c1))
            ent_acc["skip_conv_0"][bi].append(_entropy(a_c0))

            jsd_acc["skip_conv_1"][bi].append(_js_div(a_clean, a_c1))
            jsd_acc["skip_conv_0"][bi].append(_js_div(a_clean, a_c0))

            m = _ans_mass(a_clean, ans_s, ans_e)
            if not np.isnan(m):
                am_acc["clean"][bi].append(m)
                am_acc["skip_conv_1"][bi].append(_ans_mass(a_c1, ans_s, ans_e))
                am_acc["skip_conv_0"][bi].append(_ans_mass(a_c0, ans_s, ans_e))

        del sample_attn
        sample_count += 1

for h in handles:
    h.remove()
print(f"Processed {sample_count} samples. Hooks removed.\n")

# ── Results tables ──
print(f"{'='*70}")
print("ATTENTION PATTERN ANALYSIS")
print(f"{'='*70}")

print(f"\n--- Attention Entropy (higher = more uniform = degraded) ---")
print(f"{'Block':<8s}  {'Clean':>8s}  {'-Conv_1':>8s}  {'-Conv_0':>8s}  {'dE(-C1)':>8s}  {'dE(-C0)':>8s}")
print("-" * 50)
ent_means = {c: [] for c in COND_NAMES}
for bi in range(7):
    row = {}
    for c in COND_NAMES:
        row[c] = np.mean(ent_acc[c][bi]) if ent_acc[c][bi] else 0
        ent_means[c].append(row[c])
    dc1 = row["skip_conv_1"] - row["clean"]
    dc0 = row["skip_conv_0"] - row["clean"]
    print(f"Block {bi}   {row['clean']:8.3f}  {row['skip_conv_1']:8.3f}  {row['skip_conv_0']:8.3f}  {dc1:+8.3f}  {dc0:+8.3f}")
avg_ent = {c: np.mean(ent_means[c]) for c in COND_NAMES}
avg_dc1 = avg_ent["skip_conv_1"] - avg_ent["clean"]
avg_dc0 = avg_ent["skip_conv_0"] - avg_ent["clean"]
print(f"{'Avg':<8s}  {avg_ent['clean']:8.3f}  {avg_ent['skip_conv_1']:8.3f}  {avg_ent['skip_conv_0']:8.3f}  {avg_dc1:+8.3f}  {avg_dc0:+8.3f}")

print(f"\n--- JS Divergence (clean vs ablated) ---")
print(f"{'Block':<8s}  {'JSD(-C1)':>12s}  {'JSD(-C0)':>12s}  {'C1/C0':>8s}")
print("-" * 45)
jsd_means = {c: [] for c in ["skip_conv_1", "skip_conv_0"]}
for bi in range(7):
    j1 = np.mean(jsd_acc["skip_conv_1"][bi]) if jsd_acc["skip_conv_1"][bi] else 0
    j0 = np.mean(jsd_acc["skip_conv_0"][bi]) if jsd_acc["skip_conv_0"][bi] else 0
    jsd_means["skip_conv_1"].append(j1)
    jsd_means["skip_conv_0"].append(j0)
    r = j1 / max(j0, 1e-8)
    print(f"Block {bi}   {j1:12.4f}  {j0:12.4f}  {r:7.1f}x")
avg_j1 = np.mean(jsd_means["skip_conv_1"]); avg_j0 = np.mean(jsd_means["skip_conv_0"])
print(f"{'Avg':<8s}  {avg_j1:12.4f}  {avg_j0:12.4f}  {avg_j1/max(avg_j0,1e-8):7.1f}x")

print(f"\n--- Answer-Span Attention Mass ---")
print(f"{'Block':<8s}  {'Clean':>8s}  {'-Conv_1':>8s}  {'-Conv_0':>8s}  {'d(-C1)':>8s}  {'d(-C0)':>8s}")
print("-" * 50)
am_means = {c: [] for c in COND_NAMES}
for bi in range(7):
    row = {}
    for c in COND_NAMES:
        row[c] = np.mean(am_acc[c][bi]) if am_acc[c][bi] else 0
        am_means[c].append(row[c])
    dc1 = row["skip_conv_1"] - row["clean"]
    dc0 = row["skip_conv_0"] - row["clean"]
    print(f"Block {bi}   {row['clean']:8.4f}  {row['skip_conv_1']:8.4f}  {row['skip_conv_0']:8.4f}  {dc1:+8.4f}  {dc0:+8.4f}")
avg_am = {c: np.mean(am_means[c]) for c in COND_NAMES}
avg_am_dc1 = avg_am["skip_conv_1"] - avg_am["clean"]
avg_am_dc0 = avg_am["skip_conv_0"] - avg_am["clean"]
print(f"{'Avg':<8s}  {avg_am['clean']:8.4f}  {avg_am['skip_conv_1']:8.4f}  {avg_am['skip_conv_0']:8.4f}  {avg_am_dc1:+8.4f}  {avg_am_dc0:+8.4f}")

# ── Save ──
attn_results = {
    "n_samples": sample_count,
    "entropy": {"clean": ent_means["clean"], "skip_conv_1": ent_means["skip_conv_1"],
                "skip_conv_0": ent_means["skip_conv_0"], "avg_delta_c1": avg_dc1, "avg_delta_c0": avg_dc0},
    "js_divergence": {"skip_conv_1": jsd_means["skip_conv_1"], "skip_conv_0": jsd_means["skip_conv_0"]},
    "answer_mass": {"clean": am_means["clean"], "skip_conv_1": am_means["skip_conv_1"],
                    "skip_conv_0": am_means["skip_conv_0"]},
}
os.makedirs("experiments/results/H1", exist_ok=True)
with open("experiments/results/H1/h1_attn_degradation.json", "w") as f:
    json.dump(attn_results, f, indent=2)

# ── Interpretation ──
print(f"\n{'='*70}")
print("INTERPRETATION")
print(f"{'='*70}")
print(f"\n1. Entropy (attention focus):")
print(f"   -Conv_1: avg dEntropy = {avg_dc1:+.3f}")
print(f"   -Conv_0: avg dEntropy = {avg_dc0:+.3f}")
if avg_dc1 > avg_dc0 + 0.1:
    print(f"   -> Conv_1 removal causes SIGNIFICANTLY more attention degradation")
    print(f"      POSITIVE EVIDENCE: Conv_1 provides the features attention relies on.")
elif avg_dc1 > avg_dc0:
    print(f"   -> Conv_1 removal causes MORE degradation, but gap is modest.")
else:
    print(f"   -> Similar effect — Conv_1's role may not be attention-specific.")

print(f"\n2. JS divergence (pattern distortion):")
print(f"   -Conv_1: avg JSD = {avg_j1:.4f}")
print(f"   -Conv_0: avg JSD = {avg_j0:.4f}")
print(f"   Ratio: {avg_j1/max(avg_j0,1e-8):.1f}x")
if avg_j1 > 2 * avg_j0:
    print(f"   -> Conv_1 removal distorts attention {avg_j1/max(avg_j0,1e-8):.0f}x more than Conv_0")
    print(f"      Conv_1 is the PRIMARY INPUT shaping attention patterns.")

print(f"\n3. Answer attention mass:")
print(f"   Clean:   {avg_am['clean']:.4f}")
print(f"   -Conv_1: {avg_am['skip_conv_1']:.4f} (d={avg_am_dc1:+.4f})")
print(f"   -Conv_0: {avg_am['skip_conv_0']:.4f} (d={avg_am_dc0:+.4f})")
if avg_am_dc1 < avg_am_dc0 - 0.001:
    print(f"   -> Without Conv_1, attention loses focus on answer tokens.")

print(f"\n--- Causal Mechanism Chain ---")
print("  Conv_1 extracts local features")
print("    -> Self-Attention uses these to compute focused patterns")
print("    -> Removing Conv_1 degrades attention (higher entropy, distorted patterns)")
print("    -> Degraded attention -> wrong predictions")
print("  Establishes Conv_1 as the LOCAL FEATURE SUBSTRATE for attention.")

print(f"\n--- Limitations ---")
print(f"  1. {sample_count} samples; larger N would tighten estimates.")
print(f"  2. Hooks capture last call per block (pass 2 / M3).")
print(f"  3. Entropy increase is necessary but not sufficient: Conv_1 may also")
print(f"     contribute directly to residual stream beyond feeding attention.")

In [ ]:
## H1 — Representational Discriminativity: What Conv_1 ACTUALLY produces
##
## Direct measurement of Conv_1's output properties vs Conv_0 (matched control).
## If Conv_1 produces more discriminative representations (higher token-to-token
## variance, lower adjacent-token similarity, higher effective rank), this directly
## explains WHY attention depends on it.

import importlib, experiments.tracer
importlib.reload(experiments.tracer)
from experiments.tracer import qanet_forward

print("=" * 70)
print("H1: REPRESENTATIONAL DISCRIMINATIVITY — What does Conv_1 produce?")
print("=" * 70)

N_DISC = 200
comp_names = ["conv_0", "conv_1", "self_attn", "ffn"]
PROBE_LOCS = [(0, 0), (0, 3), (0, 6), (1, 0), (2, 0)]

# Accumulators: {(pass, block, comp): list of per-sample metric values}
token_var_acc = {c: {loc: [] for loc in PROBE_LOCS} for c in comp_names}
local_contrast_acc = {c: {loc: [] for loc in PROBE_LOCS} for c in comp_names}
eff_rank_acc = {c: {loc: [] for loc in PROBE_LOCS} for c in comp_names}
norm_std_acc = {c: {loc: [] for loc in PROBE_LOCS} for c in comp_names}

loader_disc = make_loader(dataset, batch_size=1, shuffle=True)
random.seed(77); torch.manual_seed(77)

n_done = 0
with torch.no_grad():
    for Cwid, Ccid, Qwid, Qcid, y1, y2, ids in tqdm(loader_disc, total=N_DISC, desc="Discriminativity"):
        if n_done >= N_DISC:
            break
        Cwid, Ccid = Cwid.to(DEVICE), Ccid.to(DEVICE)
        Qwid, Qcid = Qwid.to(DEVICE), Qcid.to(DEVICE)

        _, _, acts, _ = qanet_forward(model, Cwid, Ccid, Qwid, Qcid, collect=True)
        cmask = (Cwid == 0).squeeze(0)
        valid = (~cmask).sum().item()

        model_enc = acts.get("model_enc", {})
        for (pi, bi) in PROBE_LOCS:
            blk = model_enc.get(f"pass_{pi}", {}).get(f"block_{bi}", {})
            for c in comp_names:
                out = blk.get(c)
                if out is None:
                    continue
                # out: [1, d_model, L] → [L, d_model] (valid tokens only)
                v = out[0, :, :valid].transpose(0, 1).float()  # [L_valid, d]

                # 1. Token variance: var of L2 norms across positions
                norms = v.norm(dim=1)  # [L_valid]
                token_var_acc[c][(pi, bi)].append(norms.var().item())

                # 2. Norm std (normalized): std/mean of token norms
                mn = norms.mean().item()
                norm_std_acc[c][(pi, bi)].append(norms.std().item() / max(mn, 1e-8))

                # 3. Local contrast: 1 - avg cosine(t, t+1)
                if v.shape[0] > 1:
                    cos = F.cosine_similarity(v[:-1], v[1:], dim=1)  # [L-1]
                    local_contrast_acc[c][(pi, bi)].append((1.0 - cos.mean()).item())

                # 4. Effective rank (via singular values)
                try:
                    S = torch.linalg.svdvals(v)  # [min(L, d)]
                    S = S / S.sum().clamp(min=1e-8)
                    ent = -(S * (S + 1e-10).log()).sum()
                    eff_rank_acc[c][(pi, bi)].append(ent.exp().item())
                except:
                    pass

        n_done += 1

print(f"\nProcessed {n_done} samples.\n")

# ── Results ──
print(f"{'='*70}")
print("REPRESENTATIONAL DISCRIMINATIVITY RESULTS")
print(f"{'='*70}")

for metric_name, acc, higher_is in [
    ("Token Norm Variance", token_var_acc, "more discriminative"),
    ("Normalized Norm Std (CoV)", norm_std_acc, "more discriminative"),
    ("Local Contrast (1−cos)", local_contrast_acc, "more discriminative"),
    ("Effective Rank", eff_rank_acc, "more information-rich"),
]:
    print(f"\n--- {metric_name} (higher = {higher_is}) ---")
    print(f"{'Location':<10s}", end="")
    for c in comp_names:
        print(f"  {c:>12s}", end="")
    print(f"  {'C1/C0':>8s}")
    print("-" * 72)
    c1_all, c0_all = [], []
    for loc in PROBE_LOCS:
        print(f"p{loc[0]}_b{loc[1]:<6d}", end="")
        vals = {}
        for c in comp_names:
            v = np.mean(acc[c][loc]) if acc[c][loc] else 0
            vals[c] = v
            print(f"  {v:12.4f}", end="")
        ratio = vals["conv_1"] / max(vals["conv_0"], 1e-8)
        print(f"  {ratio:7.1f}x")
        c1_all.append(vals["conv_1"])
        c0_all.append(vals["conv_0"])
    avg_c1 = np.mean(c1_all); avg_c0 = np.mean(c0_all)
    print(f"{'Avg':<10s}", end="")
    for c in comp_names:
        avg = np.mean([np.mean(acc[c][loc]) if acc[c][loc] else 0 for loc in PROBE_LOCS])
        print(f"  {avg:12.4f}", end="")
    print(f"  {avg_c1/max(avg_c0,1e-8):7.1f}x")

# ── Component ranking ──
print(f"\n{'='*70}")
print("COMPONENT RANKING BY DISCRIMINATIVITY (averaged across locations)")
print(f"{'='*70}")
print(f"{'Metric':<28s}  {'Conv_0':>8s}  {'Conv_1':>8s}  {'Attn':>8s}  {'FFN':>8s}  {'Winner':<10s}")
print("-" * 80)
for metric_name, acc in [
    ("Token Norm Variance", token_var_acc),
    ("Norm CoV", norm_std_acc),
    ("Local Contrast", local_contrast_acc),
    ("Effective Rank", eff_rank_acc),
]:
    avgs = {}
    for c in comp_names:
        avgs[c] = np.mean([np.mean(acc[c][loc]) if acc[c][loc] else 0 for loc in PROBE_LOCS])
    winner = max(avgs, key=avgs.get)
    print(f"{metric_name:<28s}  {avgs['conv_0']:8.4f}  {avgs['conv_1']:8.4f}  {avgs['self_attn']:8.4f}  {avgs['ffn']:8.4f}  {winner}")

# ── Save ──
disc_results = {}
for metric_name, acc in [("token_var", token_var_acc), ("norm_cov", norm_std_acc),
                          ("local_contrast", local_contrast_acc), ("eff_rank", eff_rank_acc)]:
    disc_results[metric_name] = {}
    for c in comp_names:
        disc_results[metric_name][c] = np.mean([np.mean(acc[c][loc]) if acc[c][loc] else 0 for loc in PROBE_LOCS])

os.makedirs("experiments/results/H1", exist_ok=True)
with open("experiments/results/H1/h1_discriminativity.json", "w") as f:
    json.dump(disc_results, f, indent=2)

# ── Interpretation ──
c1_contrast = disc_results["local_contrast"]["conv_1"]
c0_contrast = disc_results["local_contrast"]["conv_0"]
c1_rank = disc_results["eff_rank"]["conv_1"]
c0_rank = disc_results["eff_rank"]["conv_0"]
c1_var = disc_results["token_var"]["conv_1"]
c0_var = disc_results["token_var"]["conv_0"]

print(f"\n{'='*70}")
print("INTERPRETATION")
print(f"{'='*70}")
print(f"\nConv_1 vs Conv_0 (identical architecture, different learned function):")
print(f"  Local contrast:    Conv_1 = {c1_contrast:.4f}, Conv_0 = {c0_contrast:.4f} (ratio = {c1_contrast/max(c0_contrast,1e-8):.1f}x)")
print(f"  Token norm var:    Conv_1 = {c1_var:.4f}, Conv_0 = {c0_var:.4f} (ratio = {c1_var/max(c0_var,1e-8):.1f}x)")
print(f"  Effective rank:    Conv_1 = {c1_rank:.1f}, Conv_0 = {c0_rank:.1f} (ratio = {c1_rank/max(c0_rank,1e-8):.1f}x)")

print(f"\nThis answers 'what does Conv_1 encode':")
print(f"  Conv_1 transforms its input into HIGHLY DISCRIMINATIVE local representations:")
print(f"    - Adjacent tokens receive distinct feature vectors (high local contrast)")
print(f"    - Token-level norms vary significantly across positions (high variance)")
print(f"    - The output spans a high-dimensional subspace (high effective rank)")
print(f"  Conv_0, despite identical architecture, produces much less discriminative output.")

print(f"\nThis COMPLETES the causal mechanism chain:")
print(f"  Conv_1 creates token-discriminative features (THIS experiment)")
print(f"    -> Attention computes Q,K from discriminative features -> focused patterns")
print(f"    -> Remove Conv_1 -> features become less discriminative -> Q,K become similar")
print(f"    -> Similar Q,K -> attention collapses (Exp 5: entropy down, JSD 8.2x)")
print(f"    -> Collapsed attention -> wrong predictions (Exp 1: -32.54 F1)")
print(f"  'Local feature extraction' is now a MEASURED property, not an inference.")

In [ ]:
## H1 Method 1 — Eval-time Global Ablation
## Skip ALL conv / ALL attn / ALL ffn across Model Encoder, measure F1/EM on full dev set.
## Validates directional finding from QANet paper Table 5: Conv removal > Attn removal.
##
## Implementation: zero-out the sub-layer output BEFORE residual add.
## Effect: only the residual (skip connection) survives → removes that component's computation.

import importlib, experiments.tracer
importlib.reload(experiments.tracer)
from experiments.tracer import qanet_forward, SkipSpec
from EvaluateTools.eval_utils import convert_tokens, squad_evaluate

print("=" * 70)
print("H1 METHOD 1: EVAL-TIME GLOBAL ABLATION")
print("=" * 70)
print("Mechanism: zero-out sub-layer output before residual connection.")
print("  skip_conv    → all conv_0 & conv_1 in all 7 blocks × 3 passes = 42 conv layers zeroed")
print("  skip_conv_0  → only conv_0 in all 7 blocks × 3 passes = 21 conv_0 layers zeroed")
print("  skip_conv_1  → only conv_1 in all 7 blocks × 3 passes = 21 conv_1 layers zeroed")
print("  skip_attn    → all self_attn in all 7 blocks × 3 passes = 21 attn layers zeroed")
print("  skip_ffn     → all ffn in all 7 blocks × 3 passes = 21 ffn layers zeroed")
print("  baseline     → normal forward pass (no skip)")
print(f"Eval on full dev set ({len(dataset)} samples), batch_size=32.\n")

ABLATION_CONFIGS = {
    "baseline":    None,
    "skip_conv":   SkipSpec(global_skip={"conv"}),
    "skip_conv_0": SkipSpec(global_skip={"conv_0"}),
    "skip_conv_1": SkipSpec(global_skip={"conv_1"}),
    "skip_attn":   SkipSpec(global_skip={"self_attn"}),
    "skip_ffn":    SkipSpec(global_skip={"ffn"}),
}

ablation_results = {}
for cfg_name, skip in ABLATION_CONFIGS.items():
    loader_abl = make_loader(dataset, batch_size=32, shuffle=False)
    answers = {}
    with torch.no_grad():
        for Cwid, Ccid, Qwid, Qcid, y1, y2, ids in tqdm(loader_abl, desc=f"  {cfg_name}"):
            Cwid, Ccid = Cwid.to(DEVICE), Ccid.to(DEVICE)
            Qwid, Qcid = Qwid.to(DEVICE), Qcid.to(DEVICE)

            p1, p2, _, _ = qanet_forward(model, Cwid, Ccid, Qwid, Qcid, skip_spec=skip)

            p1_log = F.log_softmax(p1, dim=1)
            p2_log = F.log_softmax(p2, dim=1)
            outer = p1_log.unsqueeze(2) + p2_log.unsqueeze(1)
            tri = torch.triu(torch.ones(outer.size(-2), outer.size(-1), device=DEVICE, dtype=torch.bool))
            outer = outer.masked_fill(~tri, float('-inf'))
            flat = outer.view(outer.size(0), -1)
            idx = torch.argmax(flat, dim=1)
            L = p1.size(1)
            ymin = idx // L; ymax = idx % L
            ans, _ = convert_tokens(eval_file, ids.tolist(), ymin.tolist(), ymax.tolist())
            answers.update(ans)

    metrics = squad_evaluate(eval_file, answers)
    ablation_results[cfg_name] = {"f1": metrics["f1"], "em": metrics["exact_match"]}

os.makedirs("experiments/results/H1", exist_ok=True)
with open("experiments/results/H1/h1_ablation_global.json", "w") as f:
    json.dump(ablation_results, f, indent=2)

base_f1 = ablation_results["baseline"]["f1"]
base_em = ablation_results["baseline"]["em"]

print(f"\n{'='*70}")
print(f"{'Config':>14s}  {'F1':>7s}  {'EM':>7s}  {'ΔF1':>8s}  {'ΔEM':>8s}  {'%F1 lost':>9s}")
print("-" * 70)
print(f"{'baseline':>14s}  {base_f1:7.2f}  {base_em:7.2f}  {'—':>8s}  {'—':>8s}  {'—':>9s}")
for k in ["skip_conv", "skip_conv_0", "skip_conv_1", "skip_attn", "skip_ffn"]:
    r = ablation_results[k]
    df1 = r["f1"] - base_f1
    dem = r["em"] - base_em
    pct = abs(df1) / base_f1 * 100
    print(f"{k:>14s}  {r['f1']:7.2f}  {r['em']:7.2f}  {df1:+8.2f}  {dem:+8.2f}  {pct:8.1f}%")
print("-" * 70)

conv_drop  = base_f1 - ablation_results["skip_conv"]["f1"]
conv0_drop = base_f1 - ablation_results["skip_conv_0"]["f1"]
conv1_drop = base_f1 - ablation_results["skip_conv_1"]["f1"]
attn_drop  = base_f1 - ablation_results["skip_attn"]["f1"]
ffn_drop   = base_f1 - ablation_results["skip_ffn"]["f1"]

print(f"\n--- Damage Ranking ---")
ranking = sorted([
    ("skip_conv", conv_drop), ("skip_conv_0", conv0_drop), ("skip_conv_1", conv1_drop),
    ("skip_attn", attn_drop), ("skip_ffn", ffn_drop),
], key=lambda x: -x[1])
for i, (name, drop) in enumerate(ranking, 1):
    print(f"  {i}. {name:<14s}  −{drop:.2f} F1")

print(f"\n--- Conv_0 vs Conv_1 Analysis ---")
print(f"  skip_conv_0: ΔF1 = {-conv0_drop:+.2f}  (21 layers zeroed)")
print(f"  skip_conv_1: ΔF1 = {-conv1_drop:+.2f}  (21 layers zeroed)")
print(f"  Conv_1/Conv_0 damage ratio: {conv1_drop/max(conv0_drop, 0.01):.2f}x")
print(f"  skip_conv (both): ΔF1 = {-conv_drop:+.2f}")
print(f"  Additivity check: conv0_drop + conv1_drop = {conv0_drop+conv1_drop:.2f} vs conv_both_drop = {conv_drop:.2f}")
interaction = conv_drop - (conv0_drop + conv1_drop)
print(f"  Interaction term: {interaction:+.2f} ({'super-additive → conv_0 & conv_1 interact' if interaction > 0.5 else 'sub-additive → partial redundancy' if interaction < -0.5 else 'roughly additive'})")
print(f"\n  Method 2 (Causal Tracing) found conv_1/conv_0 AIE ratio ≈ 4.5x")
print(f"  Method 1 (Global Ablation) finds conv_1/conv_0 ratio = {conv1_drop/max(conv0_drop, 0.01):.2f}x")
print(f"  → {'Consistent direction: conv_1 > conv_0 in both methods' if conv1_drop > conv0_drop else 'Inconsistent — needs investigation'}")

print(f"\n--- Conv vs Attn Overall ---")
print(f"  Conv(all)/Attn damage ratio: {conv_drop/max(attn_drop, 0.01):.2f}x")
print(f"  Conv_1 alone vs Attn: {conv1_drop/max(attn_drop, 0.01):.2f}x")
print(f"\nQANet paper Table 5 reference: Conv removal −2.7 F1, Attn removal −1.3 F1 (ratio ≈ 2.08x)")
print(f"Our eval-time result:          Conv removal −{conv_drop:.1f} F1, Attn removal −{attn_drop:.1f} F1 (ratio ≈ {conv_drop/max(attn_drop, 0.01):.2f}x)")
print(f"NOTE: Our drops are larger because eval-time skip ≠ retrain ablation (distribution shift).")

In [ ]:
## H1 Method 3 — Per-block Ablation
## Skip conv or attn in ONE specific (pass, block) at a time, measure F1.
## 7 blocks × 3 passes × 2 types = 42 configs.
## "conv" skip zeroes conv_0 AND conv_1 together within one block.

MAX_BATCHES_M3 = None  # None = full dev set (10465 samples, ~1.5 min/config, ~63 min total)

n_samples_est = min(len(dataset), MAX_BATCHES_M3 * 32) if MAX_BATCHES_M3 else len(dataset)
print("=" * 70)
print("H1 METHOD 3: PER-BLOCK ABLATION")
print("=" * 70)
print("Mechanism: zero-out conv (both conv_0 & conv_1) or self_attn in a SINGLE (pass, block).")
print("Everything else remains intact → isolates one block's contribution.")
print(f"Total configs: 7 blocks × 3 passes × 2 types = 42")
print(f"Eval on {'~' + str(n_samples_est) + ' subsampled' if MAX_BATCHES_M3 else 'full ' + str(len(dataset))} samples, batch_size=32.")
print(f"Baseline F1 = {base_f1:.2f}, EM = {base_em:.2f}")
if MAX_BATCHES_M3:
    print(f"NOTE: Using {MAX_BATCHES_M3} batches/config for speed. F1 variance ~±1 vs full eval.")
    print("Running subsampled baseline first...\n")

# Run baseline on the SAME subset so ΔF1 is fair
loader_base = make_loader(dataset, batch_size=32, shuffle=False)
answers_base = {}
with torch.no_grad():
    for batch_i, (Cwid, Ccid, Qwid, Qcid, y1, y2, ids) in enumerate(loader_base):
        if MAX_BATCHES_M3 and batch_i >= MAX_BATCHES_M3:
            break
        Cwid, Ccid = Cwid.to(DEVICE), Ccid.to(DEVICE)
        Qwid, Qcid = Qwid.to(DEVICE), Qcid.to(DEVICE)
        p1, p2, _, _ = qanet_forward(model, Cwid, Ccid, Qwid, Qcid, skip_spec=None)
        p1_log = F.log_softmax(p1, dim=1)
        p2_log = F.log_softmax(p2, dim=1)
        outer = p1_log.unsqueeze(2) + p2_log.unsqueeze(1)
        mask_tri = torch.triu(torch.ones(outer.size(-2), outer.size(-1), device=DEVICE, dtype=torch.bool))
        outer = outer.masked_fill(~mask_tri, float('-inf'))
        flat = outer.view(outer.size(0), -1)
        idx = torch.argmax(flat, dim=1)
        L = p1.size(1)
        ymin = idx // L; ymax = idx % L
        ans, _ = convert_tokens(eval_file, ids.tolist(), ymin.tolist(), ymax.tolist())
        answers_base.update(ans)
m3_base = squad_evaluate(eval_file, answers_base)
m3_base_f1 = m3_base["f1"]
m3_base_em = m3_base["exact_match"]
print(f"Subsampled baseline ({len(answers_base)} samples): F1 = {m3_base_f1:.2f}, EM = {m3_base_em:.2f}")
print(f"Full dev baseline (Method 1):                      F1 = {base_f1:.2f}, EM = {base_em:.2f}")
print(f"Subset-vs-full difference: ΔF1 = {m3_base_f1 - base_f1:+.2f}\n")

block_ablation = {}
total_configs = 42

for pass_idx in range(3):
    for block_idx in range(7):
        for comp_type in ["conv", "self_attn"]:
            key = f"p{pass_idx}_b{block_idx}_{comp_type}"
            skip = SkipSpec(block_skip=(pass_idx, block_idx, comp_type))
            loader_ba = make_loader(dataset, batch_size=32, shuffle=False)
            answers = {}
            with torch.no_grad():
                for batch_i, (Cwid, Ccid, Qwid, Qcid, y1, y2, ids) in enumerate(loader_ba):
                    if MAX_BATCHES_M3 and batch_i >= MAX_BATCHES_M3:
                        break
                    Cwid, Ccid = Cwid.to(DEVICE), Ccid.to(DEVICE)
                    Qwid, Qcid = Qwid.to(DEVICE), Qcid.to(DEVICE)

                    p1, p2, _, _ = qanet_forward(model, Cwid, Ccid, Qwid, Qcid, skip_spec=skip)

                    p1_log = F.log_softmax(p1, dim=1)
                    p2_log = F.log_softmax(p2, dim=1)
                    outer = p1_log.unsqueeze(2) + p2_log.unsqueeze(1)
                    mask_tri = torch.triu(torch.ones(outer.size(-2), outer.size(-1), device=DEVICE, dtype=torch.bool))
                    outer = outer.masked_fill(~mask_tri, float('-inf'))
                    flat = outer.view(outer.size(0), -1)
                    idx = torch.argmax(flat, dim=1)
                    L = p1.size(1)
                    ymin = idx // L; ymax = idx % L
                    ans, _ = convert_tokens(eval_file, ids.tolist(), ymin.tolist(), ymax.tolist())
                    answers.update(ans)

            metrics = squad_evaluate(eval_file, answers)
            m_f1 = metrics["f1"]
            m_em = metrics["exact_match"]
            block_ablation[key] = {"f1": m_f1, "em": m_em}

            done = len(block_ablation)
            df1 = m_f1 - m3_base_f1
            dem = m_em - m3_base_em
            print(f"  [{done:>2d}/{total_configs}] {key:<25s} F1={m_f1:6.2f} (ΔF1={df1:+6.2f})  EM={m_em:6.2f} (ΔEM={dem:+6.2f})")

with open("experiments/results/H1/h1_ablation_perblock.json", "w") as f:
    json.dump({
        "baseline": {"f1": m3_base_f1, "em": m3_base_em, "n_samples": len(answers_base)},
        "configs": block_ablation,
    }, f, indent=2)

ref_f1 = m3_base_f1
ref_em = m3_base_em

print(f"\n{'='*70}")
print(f"PER-BLOCK ABLATION SUMMARY  (subsampled baseline F1={ref_f1:.2f}, EM={ref_em:.2f})")
print(f"{'='*70}")

print(f"\n--- Top 10 most damaging single-block ablations ---")
print(f"{'Rank':<5s} {'Config':<25s}  {'F1':>7s}  {'ΔF1':>8s}  {'EM':>7s}  {'ΔEM':>8s}")
print("-" * 65)
ranked_abl = sorted(block_ablation.items(), key=lambda x: x[1]["f1"])
for rank, (k, r) in enumerate(ranked_abl[:10], 1):
    print(f"{rank:<5d} {k:<25s}  {r['f1']:7.2f}  {r['f1']-ref_f1:+8.2f}  {r['em']:7.2f}  {r['em']-ref_em:+8.2f}")

print(f"\n--- Average ΔF1 by Component Type ---")
conv_drops = [ref_f1 - r["f1"] for k, r in block_ablation.items() if k.endswith("_conv")]
attn_drops = [ref_f1 - r["f1"] for k, r in block_ablation.items() if k.endswith("_self_attn")]
print(f"  Skip Conv:      mean ΔF1 = {-np.mean(conv_drops):+.2f}, max drop = {np.max(conv_drops):.2f}, min drop = {np.min(conv_drops):.2f}")
print(f"  Skip Self-Attn: mean ΔF1 = {-np.mean(attn_drops):+.2f}, max drop = {np.max(attn_drops):.2f}, min drop = {np.min(attn_drops):.2f}")
print(f"  Conv is on avg {np.mean(conv_drops)/max(np.mean(attn_drops),0.01):.2f}x more damaging than Attn")

print(f"\n--- Average ΔF1 by Pass ---")
for p in range(3):
    p_drops = [ref_f1 - r["f1"] for k, r in block_ablation.items() if k.startswith(f"p{p}_")]
    print(f"  Pass {p} → M{p+1}: mean ΔF1 = {-np.mean(p_drops):+.2f}, max drop = {np.max(p_drops):.2f}")

print(f"\n--- Average ΔF1 by Block Depth ---")
for b in range(7):
    b_drops = [ref_f1 - r["f1"] for k, r in block_ablation.items() if f"_b{b}_" in k]
    bar = "█" * int(max(np.mean(b_drops), 0) * 10)
    print(f"  Block {b}: mean ΔF1 = {-np.mean(b_drops):+.2f}  {bar}")

In [ ]:
## [REMOVED] H1 Cross-Validation Analysis — superseded by new 3-experiment structure
with open("experiments/results/H1/h1_results.json") as f:
    h1_trace = json.load(f)["results"]
with open("experiments/results/H1/h1_ablation_global.json") as f:
    h1_global = json.load(f)
with open("experiments/results/H1/h1_ablation_perblock.json") as f:
    h1_block_raw = json.load(f)
if "configs" in h1_block_raw:
    h1_block = h1_block_raw["configs"]
    m3_ref_f1 = h1_block_raw["baseline"]["f1"]
else:
    h1_block = h1_block_raw
    m3_ref_f1 = None
base_f1 = h1_global["baseline"]["f1"]

fig = plt.figure(figsize=(20, 14))

# ── Panel 1: Method 1 — Global ablation bars ──
ax1 = fig.add_subplot(2, 3, 1)
cfgs = ["skip_conv", "skip_conv_0", "skip_conv_1", "skip_attn", "skip_ffn"]
labels = ["Conv(all)", "Conv_0", "Conv_1", "Attn", "FFN"]
delta_f1 = [h1_global[c]["f1"] - base_f1 for c in cfgs]
colors = ["#1565C0", "#42A5F5", "#2196F3", "#FF9800", "#4CAF50"]
ax1.bar(labels, delta_f1, color=colors, edgecolor="black")
ax1.set_ylabel("ΔF1 vs Baseline")
ax1.set_title("Method 1: Global Ablation")
ax1.axhline(0, color="gray", ls="--", alpha=0.5)
for i, v in enumerate(delta_f1):
    ax1.text(i, v - 0.5, f"{v:+.1f}", ha="center", fontweight="bold", fontsize=7)

# ── Panel 2: Method 2 — Causal tracing aggregate by component type ──
ax2 = fig.add_subplot(2, 3, 2)
components = ["conv_0", "conv_1", "self_attn", "ffn"]
comp_labels = ["Conv 0", "Conv 1", "Self-Attn", "FFN"]
agg = {c: [] for c in components}
for key, val in h1_trace.items():
    for c in components:
        if key.endswith(c):
            agg[c].append(val["aie_span"])
means = [np.mean(agg[c]) for c in components]
errs = [np.std(agg[c])/np.sqrt(len(agg[c]))*1.96 if agg[c] else 0 for c in components]
ax2.bar(comp_labels, means, yerr=errs, color=["#2196F3","#42A5F5","#FF9800","#4CAF50"], capsize=4, edgecolor="black")
ax2.set_ylabel("Mean AIE (span)")
ax2.set_title("Method 2: Causal Tracing (by component)")
ax2.axhline(0, color="gray", ls="--", alpha=0.5)

# ── Panel 3: Method 2 — Causal tracing aggregate by pass ──
ax3 = fig.add_subplot(2, 3, 3)
pass_agg = {p: [] for p in range(3)}
for key, val in h1_trace.items():
    p = int(key[1])
    pass_agg[p].append(val["aie_span"])
p_means = [np.mean(pass_agg[p]) for p in range(3)]
p_errs = [np.std(pass_agg[p])/np.sqrt(len(pass_agg[p]))*1.96 for p in range(3)]
ax3.bar(["Pass 1→M1", "Pass 2→M2", "Pass 3→M3"], p_means, yerr=p_errs,
        color=["#E91E63","#9C27B0","#673AB7"], capsize=4, edgecolor="black")
ax3.set_ylabel("Mean AIE (span)")
ax3.set_title("Method 2: Causal Tracing (by pass)")

# ── Panel 4: Method 3 — Per-block ablation heatmap (ΔF1) ──
ax4 = fig.add_subplot(2, 3, 4)
pb_ref = m3_ref_f1 if m3_ref_f1 is not None else base_f1
abl_matrix = np.zeros((2, 21))
for p in range(3):
    for b in range(7):
        col = p * 7 + b
        cv = h1_block.get(f"p{p}_b{b}_conv", {}).get("f1", pb_ref) - pb_ref
        at = h1_block.get(f"p{p}_b{b}_self_attn", {}).get("f1", pb_ref) - pb_ref
        abl_matrix[0, col] = cv
        abl_matrix[1, col] = at
vmax = max(abs(abl_matrix.min()), abs(abl_matrix.max()), 0.1)
im4 = ax4.imshow(abl_matrix, cmap="RdYlBu", aspect="auto", vmin=-vmax, vmax=vmax*0.1)
ax4.set_yticks([0, 1]); ax4.set_yticklabels(["Skip Conv", "Skip Attn"])
ax4.set_xticks(range(21)); ax4.set_xticklabels([f"B{b}" for _ in range(3) for b in range(7)], fontsize=6)
for p in range(1, 3): ax4.axvline(x=p*7-0.5, color="white", lw=2)
plt.colorbar(im4, ax=ax4, label="ΔF1", shrink=0.8)
ax4.set_title("Method 3: Per-block Ablation (ΔF1)")

# ── Panel 5: Method 2 heatmap (AIE) for comparison ──
ax5 = fig.add_subplot(2, 3, 5)
trace_conv_matrix = np.zeros((2, 21))
for p in range(3):
    for b in range(7):
        col = p * 7 + b
        c0 = h1_trace.get(f"p{p}_b{b}_conv_0", {}).get("aie_span", 0)
        c1 = h1_trace.get(f"p{p}_b{b}_conv_1", {}).get("aie_span", 0)
        at = h1_trace.get(f"p{p}_b{b}_self_attn", {}).get("aie_span", 0)
        trace_conv_matrix[0, col] = c0 + c1
        trace_conv_matrix[1, col] = at
vmax5 = max(abs(trace_conv_matrix.max()), 1e-6)
im5 = ax5.imshow(trace_conv_matrix, cmap="RdYlBu_r", aspect="auto", vmin=0, vmax=vmax5)
ax5.set_yticks([0, 1]); ax5.set_yticklabels(["Conv (c0+c1)", "Self-Attn"])
ax5.set_xticks(range(21)); ax5.set_xticklabels([f"B{b}" for _ in range(3) for b in range(7)], fontsize=6)
for p in range(1, 3): ax5.axvline(x=p*7-0.5, color="white", lw=2)
plt.colorbar(im5, ax=ax5, label="AIE", shrink=0.8)
ax5.set_title("Method 2: Causal Tracing (Conv vs Attn)")

# ── Panel 6: Correlation between Method 2 and Method 3 ──
ax6 = fig.add_subplot(2, 3, 6)
m2_vals, m3_vals, point_labels = [], [], []
for p in range(3):
    for b in range(7):
        c0 = h1_trace.get(f"p{p}_b{b}_conv_0", {}).get("aie_span", 0)
        c1 = h1_trace.get(f"p{p}_b{b}_conv_1", {}).get("aie_span", 0)
        at = h1_trace.get(f"p{p}_b{b}_self_attn", {}).get("aie_span", 0)
        cv_df1 = pb_ref - h1_block.get(f"p{p}_b{b}_conv", {}).get("f1", pb_ref)
        at_df1 = pb_ref - h1_block.get(f"p{p}_b{b}_self_attn", {}).get("f1", pb_ref)
        m2_vals.extend([c0+c1, at])
        m3_vals.extend([cv_df1, at_df1])
        point_labels.extend([f"p{p}b{b}_conv", f"p{p}b{b}_attn"])
ax6.scatter(m2_vals, m3_vals, alpha=0.6, s=30)
corr = np.corrcoef(m2_vals, m3_vals)[0, 1] if len(m2_vals) > 2 else 0
ax6.set_xlabel("Method 2: AIE (causal tracing)")
ax6.set_ylabel("Method 3: ΔF1 (per-block ablation)")
ax6.set_title(f"Cross-validation: r = {corr:.3f}")
z = np.polyfit(m2_vals, m3_vals, 1)
xs = np.linspace(min(m2_vals), max(m2_vals), 50)
ax6.plot(xs, np.polyval(z, xs), "r--", alpha=0.5)

fig.suptitle("H1: Conv vs Self-Attention — Three-Method Cross-Validation", fontsize=15, fontweight="bold")
fig.tight_layout(); plt.show()

print(f"\n{'='*70}")
print("H1 CROSS-VALIDATION SUMMARY")
print(f"{'='*70}")

print(f"\n[Method 1] Global Ablation (eval-time, full dev set):")
for c in ["skip_conv", "skip_conv_0", "skip_conv_1", "skip_attn", "skip_ffn"]:
    r = h1_global[c]
    print(f"  {c:>14s}: F1={r['f1']:.2f}  ΔF1={r['f1']-base_f1:+.2f}")
conv_g = base_f1 - h1_global["skip_conv"]["f1"]
conv0_g = base_f1 - h1_global["skip_conv_0"]["f1"]
conv1_g = base_f1 - h1_global["skip_conv_1"]["f1"]
attn_g = base_f1 - h1_global["skip_attn"]["f1"]
print(f"  → Conv(all)/Attn ratio = {conv_g/max(attn_g,0.01):.2f}x, Conv_1/Conv_0 ratio = {conv1_g/max(conv0_g,0.01):.2f}x")

print(f"\n[Method 2] ROME-style Causal Tracing (AIE):")
conv_aie = sum(h1_trace[k]["aie_span"] for k in h1_trace if "conv" in k)
attn_aie = sum(h1_trace[k]["aie_span"] for k in h1_trace if k.endswith("self_attn"))
print(f"  Total AIE: Conv(0+1) = {conv_aie:.4f}, Self-Attn = {attn_aie:.4f}")
print(f"  → Conv/Attn AIE ratio = {conv_aie/max(attn_aie,1e-6):.2f}x")

print(f"\n[Method 3] Per-block Ablation (42 configs, full dev set):")
conv_perblk = np.mean([pb_ref - h1_block[k]["f1"] for k in h1_block if k.endswith("_conv")])
attn_perblk = np.mean([pb_ref - h1_block[k]["f1"] for k in h1_block if k.endswith("_self_attn")])
print(f"  Avg F1 drop: Conv = {conv_perblk:.2f}, Self-Attn = {attn_perblk:.2f}")
print(f"  → Conv/Attn drop ratio = {conv_perblk/max(attn_perblk,0.01):.2f}x")

print(f"\n[Cross-method Correlation] Method 2 (AIE) vs Method 3 (ΔF1): Pearson r = {corr:.3f}")
if corr > 0.5:
    print("  Strong positive correlation: causal tracing and ablation agree on which components matter.")
elif corr > 0.2:
    print("  Moderate correlation: partial agreement between methods.")
else:
    print("  Weak correlation: methods capture different aspects of importance.")

all_three_agree = conv_g > attn_g and conv_aie > attn_aie and conv_perblk > attn_perblk
print(f"\n[Verdict] All three methods agree Conv > Attn? {'YES ✓' if all_three_agree else 'NO — see details above'}")
if all_three_agree:
    print("  The hypothesis is SUPPORTED: Conv sub-layers are causally more important than Self-Attention.")
    print(f"  Consistency ratios: M1={conv_g/max(attn_g,0.01):.2f}x, M2={conv_aie/max(attn_aie,1e-6):.2f}x, M3={conv_perblk/max(attn_perblk,0.01):.2f}x")

In [ ]:
## [REMOVED] H1 — Bootstrap Confidence Intervals (not referenced in current report)

import numpy as np

def bootstrap_ci(data, n_boot=10000, ci=0.95, seed=42):
    """Non-parametric bootstrap CI for the mean."""
    data = np.array(data, dtype=float)
    n = len(data)
    if n < 2:
        return float(np.mean(data)), float(np.mean(data)), float(np.mean(data))
    rng = np.random.RandomState(seed)
    idx = rng.randint(0, n, size=(n_boot, n))
    boot_means = data[idx].mean(axis=1)
    alpha = (1 - ci) / 2
    lo = np.percentile(boot_means, alpha * 100)
    hi = np.percentile(boot_means, (1 - alpha) * 100)
    return float(np.mean(data)), float(lo), float(hi)

print(f"{'='*80}")
print("H1: BOOTSTRAP 95% CONFIDENCE INTERVALS")
print(f"{'='*80}")
print(f"Method: non-parametric bootstrap, 10,000 resamples, seed=42\n")

# ─── 1. Causal Tracing AIE ───
print(f"{'─'*80}")
print("1. Causal Tracing — Mean AIE by Component Type")
print(f"{'─'*80}")
try:
    components_ct = ["conv_0", "conv_1", "self_attn", "ffn"]
    print(f"  Source: h1_ie (per-sample AIE, {len(h1_te_list)} samples × {NOISE_REPEATS} noise repeats)")
    print(f"  Average TE = {np.mean(h1_te_list):.4f}\n")

    print(f"  {'Component':<14s}  {'Mean AIE':>10s}  {'95% CI':>22s}  {'Width':>8s}  {'Sum AIE':>10s}")
    print(f"  {'-'*70}")

    ct_boot = {}
    for comp in components_ct:
        comp_keys = [k for k in spec_keys_h1 if k.endswith(comp)]
        per_sample_sum = np.zeros(len(h1_ie[comp_keys[0]]))
        for k in comp_keys:
            arr = np.array(h1_ie[k])
            per_sample_sum[:len(arr)] += arr[:len(per_sample_sum)]
        per_sample_mean = per_sample_sum / len(comp_keys)
        mean_v, lo, hi = bootstrap_ci(per_sample_mean)
        ct_boot[comp] = (mean_v, lo, hi)
        sum_v = mean_v * len(comp_keys)
        print(f"  {comp:<14s}  {mean_v:10.5f}  [{lo:.5f}, {hi:.5f}]  {hi-lo:8.5f}  {sum_v:10.4f}")

    c1_m, c1_lo, c1_hi = ct_boot["conv_1"]
    at_m, at_lo, at_hi = ct_boot["self_attn"]
    if c1_lo > at_hi:
        print(f"\n  ** Conv_1 AIE ({c1_lo:.5f}–{c1_hi:.5f}) does NOT overlap Attn AIE ({at_lo:.5f}–{at_hi:.5f}) **")
        print(f"  ** → Conv_1 > Attn is statistically significant at 95% level **")
    else:
        print(f"\n  Conv_1 CI: [{c1_lo:.5f}, {c1_hi:.5f}],  Attn CI: [{at_lo:.5f}, {at_hi:.5f}]")

except NameError:
    print("  [SKIP] Causal tracing variables (h1_ie) not in memory. Run Cell 17 first.")

# ─── 2. Attention JSD Ratio ───
print(f"\n{'─'*80}")
print("2. Attention Pattern Degradation — JSD Ratio")
print(f"{'─'*80}")
try:
    print(f"  Source: jsd_acc (per-sample JSD, {sample_count} samples × 7 blocks)\n")

    per_sample_jsd_c1 = []
    per_sample_jsd_c0 = []
    for bi in range(7):
        per_sample_jsd_c1.extend(jsd_acc["skip_conv_1"][bi])
        per_sample_jsd_c0.extend(jsd_acc["skip_conv_0"][bi])

    m1, lo1, hi1 = bootstrap_ci(per_sample_jsd_c1)
    m0, lo0, hi0 = bootstrap_ci(per_sample_jsd_c0)
    print(f"  JSD(−Conv_1): mean = {m1:.5f}, 95% CI = [{lo1:.5f}, {hi1:.5f}]")
    print(f"  JSD(−Conv_0): mean = {m0:.5f}, 95% CI = [{lo0:.5f}, {hi0:.5f}]")

    ratios = np.array(per_sample_jsd_c1) / np.maximum(np.array(per_sample_jsd_c0), 1e-10)
    rm, rlo, rhi = bootstrap_ci(ratios)
    print(f"\n  Per-sample ratio JSD(−C1)/JSD(−C0):")
    print(f"    Mean = {rm:.2f}×, 95% CI = [{rlo:.2f}×, {rhi:.2f}×]")

    if rlo > 1.0:
        print(f"  ** Ratio CI entirely above 1.0 → Conv_1 causes significantly more attention distortion **")

    print(f"\n  Per-block JSD with bootstrap CI:")
    print(f"  {'Block':<8s}  {'JSD(−C1)':>10s}  {'CI_C1':>22s}  {'JSD(−C0)':>10s}  {'CI_C0':>22s}")
    print(f"  {'-'*76}")
    for bi in range(7):
        m1b, lo1b, hi1b = bootstrap_ci(jsd_acc["skip_conv_1"][bi])
        m0b, lo0b, hi0b = bootstrap_ci(jsd_acc["skip_conv_0"][bi])
        print(f"  Block {bi}   {m1b:10.4f}  [{lo1b:.4f}, {hi1b:.4f}]  {m0b:10.4f}  [{lo0b:.4f}, {hi0b:.4f}]")

except NameError:
    print("  [SKIP] Attention degradation variables (jsd_acc) not in memory. Run Cell 24 first.")

# ─── 3. Causal Tracing TE ───
print(f"\n{'─'*80}")
print("3. Total Effect (TE) — Noise Corruption Effectiveness")
print(f"{'─'*80}")
try:
    te_m, te_lo, te_hi = bootstrap_ci(h1_te_list)
    print(f"  TE: mean = {te_m:.4f}, 95% CI = [{te_lo:.4f}, {te_hi:.4f}]")
    if te_lo > 0:
        print(f"  ** CI entirely above 0 → noise corruption reliably degrades performance **")
except NameError:
    print("  [SKIP] h1_te_list not in memory.")

print(f"\n{'='*80}")
print("SUMMARY: Statistical Significance of Key H1 Claims")
print(f"{'='*80}")
try:
    claims = []
    c1_m, c1_lo, c1_hi = ct_boot["conv_1"]
    at_m, at_lo, at_hi = ct_boot["self_attn"]
    claims.append(("Conv_1 AIE > Attn AIE", c1_lo > at_hi, f"C1=[{c1_lo:.5f},{c1_hi:.5f}] vs A=[{at_lo:.5f},{at_hi:.5f}]"))

    c0_m, c0_lo, c0_hi = ct_boot["conv_0"]
    claims.append(("Conv_1 AIE > Conv_0 AIE", c1_lo > c0_hi, f"C1=[{c1_lo:.5f},{c1_hi:.5f}] vs C0=[{c0_lo:.5f},{c0_hi:.5f}]"))

    claims.append(("JSD ratio > 1 (Conv_1 distorts attn more)", rlo > 1.0, f"ratio CI = [{rlo:.2f}, {rhi:.2f}]"))
    claims.append(("TE > 0 (corruption works)", te_lo > 0, f"TE CI = [{te_lo:.4f}, {te_hi:.4f}]"))

    for claim, sig, detail in claims:
        status = "SIGNIFICANT" if sig else "not significant"
        print(f"  {'✓' if sig else '✗'} {claim}: {status}")
        print(f"    {detail}")
except:
    print("  [Some variables not available — run all H1 cells first]")

In [ ]:
## [REMOVED] H1 — Old Summary Visualizations (to be replaced with new report-aligned figures)
## Fig 4: Attention JSD — Conv_1 vs Conv_0 per block

import matplotlib.pyplot as plt
import numpy as np

fig_num = 0

# ═══════════════════ Fig 1: Global Ablation ═══════════════════
try:
    fig_num += 1
    fig, ax = plt.subplots(figsize=(8, 5))
    comps = ["Conv_0", "Conv_1", "Self-Attn", "FFN", "Conv(both)"]
    base = ablation_results["baseline"]
    base_f1 = base["f1"]
    deltas = []
    for cfg in ["skip_conv_0", "skip_conv_1", "skip_attn", "skip_ffn", "skip_conv"]:
        deltas.append(ablation_results[cfg]["f1"] - base_f1)
    colors = ["#64B5F6", "#1565C0", "#FF9800", "#4CAF50", "#B71C1C"]
    bars = ax.bar(comps, deltas, color=colors, edgecolor="black", linewidth=0.8)
    for bar, d in zip(bars, deltas):
        ax.text(bar.get_x() + bar.get_width()/2, d - 1.5, f"{d:.1f}",
                ha="center", va="top", fontsize=11, fontweight="bold", color="white")
    ax.set_ylabel("ΔF1 (vs baseline)", fontsize=12)
    ax.set_title(f"Fig {fig_num}: Global Ablation Damage (baseline F1 = {base_f1:.2f})", fontsize=13)
    ax.axhline(0, color="gray", ls="--", alpha=0.5)
    ax.set_ylim(min(deltas) * 1.15, 2)
    fig.tight_layout(); plt.show()
    print(f"Fig {fig_num}: Conv_1 removal (−{abs(deltas[1]):.1f} F1) is {abs(deltas[1])/max(abs(deltas[2]),0.01):.0f}× "
          f"more damaging than Self-Attn removal (−{abs(deltas[2]):.1f} F1)")
except Exception as e:
    print(f"[Fig 1 skipped: {e}]")

# ═══════════════════ Fig 2: Multi-property Probe AUC ═══════════════════
try:
    fig_num += 1
    comp_list = ["conv_0", "conv_1", "self_attn", "ffn"]
    comp_display = ["Conv_0", "Conv_1", "Self-Attn", "FFN"]
    ans_aucs = [np.mean([v["auc"] for k, v in func_results.items() if k[2] == c and k[3] == "is_answer"]) for c in comp_list]
    ovl_aucs = [np.mean([v["auc"] for k, v in func_results.items() if k[2] == c and k[3] == "is_overlap"]) for c in comp_list]

    x = np.arange(len(comp_list))
    w = 0.35
    fig, ax = plt.subplots(figsize=(8, 5))
    b1 = ax.bar(x - w/2, ans_aucs, w, label="is_answer", color="#1565C0", edgecolor="black", linewidth=0.8)
    b2 = ax.bar(x + w/2, ovl_aucs, w, label="is_overlap (Q∩C)", color="#FF9800", edgecolor="black", linewidth=0.8)
    for bars in [b1, b2]:
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.003, f"{h:.3f}",
                    ha="center", va="bottom", fontsize=9)
    ax.set_xticks(x); ax.set_xticklabels(comp_display, fontsize=11)
    ax.set_ylabel("Mean ROC-AUC", fontsize=12)
    ax.set_title(f"Fig {fig_num}: Multi-Property Probe — Answer Position vs Question-Overlap", fontsize=13)
    ax.set_ylim(0.5, 1.05)
    ax.legend(fontsize=11)
    fig.tight_layout(); plt.show()

    gap = max(ovl_aucs) - min(ovl_aucs)
    print(f"Fig {fig_num}: Overlap AUC gap across components = {gap:.4f}. "
          f"{'Meaningful difference detected.' if gap > 0.02 else 'All components encode overlap similarly.'}")
except Exception as e:
    print(f"[Fig 2 skipped: {e}]")

# ═══════════════════ Fig 3: Local Coherence ═══════════════════
try:
    fig_num += 1
    lags = [1, 2, 3, 5]
    fig, ax = plt.subplots(figsize=(8, 5))
    markers = {"conv_0": "s", "conv_1": "D", "self_attn": "o", "ffn": "^"}
    colors_c = {"conv_0": "#64B5F6", "conv_1": "#1565C0", "self_attn": "#FF9800", "ffn": "#4CAF50"}
    for c, disp in zip(comp_list, comp_display):
        vals = [np.mean(coherence[c][lag]) if coherence[c][lag] else 0 for lag in lags]
        ax.plot(lags, vals, marker=markers[c], label=disp, color=colors_c[c],
                linewidth=2, markersize=8)
    ax.set_xlabel("Lag (token distance)", fontsize=12)
    ax.set_ylabel("Mean Cosine Similarity", fontsize=12)
    ax.set_title(f"Fig {fig_num}: Spatial Autocorrelation by Component (p0_b0)", fontsize=13)
    ax.set_xticks(lags)
    ax.legend(fontsize=11)
    ax.grid(alpha=0.3)
    fig.tight_layout(); plt.show()
    print(f"Fig {fig_num}: Shows how quickly each component's output decorrelates with distance.")
except Exception as e:
    print(f"[Fig 3 skipped: {e}]")

# ═══════════════════ Fig 4: Attention JSD per Block ═══════════════════
try:
    fig_num += 1
    blocks = list(range(7))
    j1_vals = [np.mean(jsd_acc["skip_conv_1"][bi]) for bi in blocks]
    j0_vals = [np.mean(jsd_acc["skip_conv_0"][bi]) for bi in blocks]

    x = np.arange(7)
    w = 0.35
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(x - w/2, j1_vals, w, label="−Conv_1", color="#D32F2F", edgecolor="black", linewidth=0.8)
    ax.bar(x + w/2, j0_vals, w, label="−Conv_0 (control)", color="#64B5F6", edgecolor="black", linewidth=0.8)
    ax.set_xticks(x); ax.set_xticklabels([f"Block {b}" for b in blocks])
    ax.set_ylabel("JS Divergence vs Clean", fontsize=12)
    ax.set_title(f"Fig {fig_num}: Attention Pattern Distortion — Conv_1 vs Conv_0 Removal", fontsize=13)
    ax.legend(fontsize=11)
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout(); plt.show()

    avg_ratio = np.mean(j1_vals) / max(np.mean(j0_vals), 1e-8)
    print(f"Fig {fig_num}: Conv_1 removal causes {avg_ratio:.1f}× more attention distortion than Conv_0 (matched control).")
except Exception as e:
    print(f"[Fig 4 skipped: {e}]")

### H2: Context vs Question Dual-Stream Information Flow

**Hypothesis**: Question corruption is more destructive than Context corruption. CQ Attention is the critical information bottleneck (recovering its output restores >70% of TE).

**Method**: Three corruption conditions (C / Q / Both) × restore Embedding Encoder sub-layers and CQ Attention.

In [ ]:
from experiments.tracer import build_emb_enc_specs, build_fusion_specs

NUM_SAMPLES_H2 = 200
NOISE_REPEATS_H2 = 3

random.seed(SEED); torch.manual_seed(SEED)
loader_h2 = make_loader(dataset, batch_size=1, shuffle=True)

emb_specs = build_emb_enc_specs()
fusion_specs = build_fusion_specs()
all_specs_h2 = emb_specs + fusion_specs
all_keys_h2 = [f"{s.stage}_{s.component}" for s in all_specs_h2]

corrupt_conditions = ["context", "question", "both"]
h2_te = {cc: [] for cc in corrupt_conditions}
h2_ie = {cc: {k: [] for k in all_keys_h2} for cc in corrupt_conditions}

samples_h2 = 0
pbar_h2 = tqdm(total=NUM_SAMPLES_H2, desc="H2 Dual-Stream")
with torch.no_grad():
    for batch_idx, (Cwid, Ccid, Qwid, Qcid, y1, y2, ids) in enumerate(loader_h2):
        if samples_h2 >= NUM_SAMPLES_H2:
            break
        Cwid, Ccid = Cwid.to(DEVICE), Ccid.to(DEVICE)
        Qwid, Qcid = Qwid.to(DEVICE), Qcid.to(DEVICE)
        y1, y2 = y1.to(DEVICE), y2.to(DEVICE)

        p1_c, p2_c, clean_acts, _ = qanet_forward(model, Cwid, Ccid, Qwid, Qcid, collect=True)
        prob_clean = compute_span_prob(p1_c, p2_c, y1, y2)
        if prob_clean.item() < MIN_CLEAN_PROB:
            continue

        for cc in corrupt_conditions:
            reps_te, reps_ie = [], {k: [] for k in all_keys_h2}
            for rep in range(NOISE_REPEATS_H2):
                ns = SEED + batch_idx * 100 + rep
                p1_x, p2_x, _, _ = qanet_forward(model, Cwid, Ccid, Qwid, Qcid,
                                                  corrupt_target=cc, noise_std_scale=NOISE_STD_SCALE, noise_seed=ns)
                prob_x = compute_span_prob(p1_x, p2_x, y1, y2)
                te = (prob_clean - prob_x).item()
                reps_te.append(te)
                if abs(te) < 1e-6:
                    continue

                for spec, key in zip(all_specs_h2, all_keys_h2):
                    if cc == "context" and spec.stage == "emb_enc_Q":
                        continue
                    if cc == "question" and spec.stage == "emb_enc_C":
                        continue
                    p1_r, p2_r, _, _ = qanet_forward(model, Cwid, Ccid, Qwid, Qcid,
                                                      corrupt_target=cc, noise_std_scale=NOISE_STD_SCALE, noise_seed=ns,
                                                      clean_acts=clean_acts, restore_spec=spec)
                    prob_r = compute_span_prob(p1_r, p2_r, y1, y2)
                    reps_ie[key].append((prob_r - prob_x).item())

            h2_te[cc].append(np.mean(reps_te) if reps_te else 0.0)
            for key in all_keys_h2:
                if reps_ie[key]:
                    h2_ie[cc][key].append(np.mean(reps_ie[key]))

        samples_h2 += 1
        pbar_h2.update(1)
        pbar_h2.set_postfix(
            TE_C=f"{np.mean(h2_te['context']):.3f}",
            TE_Q=f"{np.mean(h2_te['question']):.3f}",
            TE_CQ=f"{np.mean(h2_te['both']):.3f}",
        )
pbar_h2.close()

# Build results dict
h2_results = {}
for cc in corrupt_conditions:
    te_arr = np.array(h2_te[cc])
    n_te = len(te_arr)
    h2_results[cc] = {
        "te_mean": float(te_arr.mean()) if n_te else 0,
        "te_ci95": float(te_arr.std()*1.96/np.sqrt(n_te)) if n_te>1 else 0,
        "ie": {},
    }
    for key in all_keys_h2:
        vals = np.array(h2_ie[cc].get(key, []))
        n = len(vals)
        if n > 0:
            h2_results[cc]["ie"][key] = {
                "aie": float(vals.mean()),
                "ci95": float(vals.std()*1.96/np.sqrt(n)) if n>1 else 0,
                "nie": float(vals.mean() / max(abs(te_arr.mean()), 1e-8)),
            }

os.makedirs("experiments/results/H2", exist_ok=True)
with open("experiments/results/H2/h2_results.json", "w") as f:
    json.dump(h2_results, f, indent=2)

print(f"\n{'='*70}")
print("H2: DUAL-STREAM INFORMATION FLOW — RESULTS")
print(f"{'='*70}")
print(f"Samples: {samples_h2}  |  Noise repeats: {NOISE_REPEATS_H2}  |  σ scale: {NOISE_STD_SCALE}")
print(f"Corruption targets: context (word embeddings), question (word embeddings), both")
print(f"Restoration points: Embedding Encoder sub-layers + CQ Attention + CQ Resizer\n")

print(f"--- Total Effect (TE) by Corruption Target ---")
print(f"{'Target':>10s}  {'Mean TE':>10s}  {'±95%CI':>10s}  {'Interpretation'}")
print("-" * 65)
for cc in corrupt_conditions:
    te = h2_results[cc]["te_mean"]
    ci = h2_results[cc]["te_ci95"]
    interp = "(context contribution)" if cc == "context" else ("(question contribution)" if cc == "question" else "(joint, may overlap)")
    print(f"{cc:>10s}  {te:10.4f}  {ci:10.4f}  {interp}")

te_c = h2_results["context"]["te_mean"]
te_q = h2_results["question"]["te_mean"]
te_b = h2_results["both"]["te_mean"]
print(f"\nTE(context) + TE(question) = {te_c+te_q:.4f} vs TE(both) = {te_b:.4f}")
print(f"Interaction gap = {te_b - (te_c + te_q):.4f}  "
      f"({'sub-additive → info overlap' if te_b < te_c + te_q else 'super-additive → synergy'})")
print(f"Context/Question TE ratio = {te_c/max(te_q,1e-6):.2f}x")

print(f"\n--- Indirect Effect (IE) by Restoration Point ---")
for cc in corrupt_conditions:
    print(f"\n  Corrupt: {cc}")
    ie_dict = h2_results[cc]["ie"]
    te_val = h2_results[cc]["te_mean"]
    if not ie_dict:
        print("    (no valid IE measurements)")
        continue
    print(f"  {'Restore point':<30s}  {'AIE':>8s}  {'NIE':>8s}  {'95%CI':>8s}  {'% of TE':>8s}")
    print(f"  {'-'*62}")
    for key in sorted(ie_dict.keys(), key=lambda k: -ie_dict[k]["aie"]):
        v = ie_dict[key]
        pct = v["aie"] / max(abs(te_val), 1e-8) * 100
        label = key.replace("emb_enc_C_", "EmbEnc-C: ").replace("emb_enc_Q_", "EmbEnc-Q: ") \
                    .replace("cq_att_", "CQ-Att: ").replace("cq_resizer_", "CQ-Resize: ")
        print(f"  {label:<30s}  {v['aie']:8.4f}  {v['nie']:8.4f}  {v['ci95']:8.4f}  {pct:7.1f}%")

print(f"\n--- CQ Attention Bottleneck Analysis ---")
for cc in corrupt_conditions:
    cq = h2_results[cc]["ie"].get("cq_att_output", {})
    te_val = h2_results[cc]["te_mean"]
    if cq:
        recovery = cq["nie"]
        print(f"  {cc:>10s}: CQ-Att restores {recovery:.1%} of TE ({cq['aie']:.4f} / {te_val:.4f})")
    else:
        print(f"  {cc:>10s}: CQ-Att data not available")

print(f"\n[Hypothesis check] CQ Attention recovery >70% for all conditions?")
all_above = all(
    h2_results[cc]["ie"].get("cq_att_output", {}).get("nie", 0) > 0.7
    for cc in corrupt_conditions
)
print(f"  {'YES ✓ — CQ Attention is the information bottleneck' if all_above else 'NO — threshold not met in all conditions'}")

In [ ]:
## H2 — Visualization
import matplotlib.pyplot as plt

# --- 1. TE comparison across corruption targets ---
fig, ax = plt.subplots(figsize=(6, 5))
cc_labels = ["Context", "Question", "Both"]
te_vals = [h2_results[cc]["te_mean"] for cc in corrupt_conditions]
te_errs = [h2_results[cc]["te_ci95"] for cc in corrupt_conditions]
colors_te = ["#2196F3", "#FF9800", "#F44336"]
ax.bar(cc_labels, te_vals, yerr=te_errs, color=colors_te, capsize=5, edgecolor="black")
ax.set_ylabel("Total Effect (TE)"); ax.set_title("H2: TE by Corruption Target")
fig.tight_layout(); plt.show()

# --- 2. NIE bars for each condition ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
for i, cc in enumerate(corrupt_conditions):
    ax = axes[i]
    ie_dict = h2_results[cc]["ie"]
    keys = sorted(ie_dict.keys())
    vals = [ie_dict[k]["nie"] for k in keys]
    errs = [ie_dict[k]["ci95"] / max(abs(h2_results[cc]["te_mean"]),1e-8) for k in keys]
    labels = [k.replace("emb_enc_C_", "EncC:").replace("emb_enc_Q_", "EncQ:")
               .replace("cq_att_", "CQAtt:").replace("cq_resizer_", "Resize:") for k in keys]
    c = "#2196F3" if cc=="context" else ("#FF9800" if cc=="question" else "#F44336")
    ax.barh(range(len(keys)), vals, xerr=errs, color=c, alpha=0.8, capsize=3)
    ax.set_yticks(range(len(keys))); ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel("Normalized IE"); ax.set_title(f"Corrupt: {cc_labels[i]}")
    ax.axvline(0, color="gray", ls="--", alpha=0.5)
fig.suptitle("H2: Normalized Indirect Effect by Restoration Point", fontsize=14)
fig.tight_layout(); plt.show()

# --- 3. CQ Attention bottleneck summary ---
print("\n=== CQ Attention Bottleneck Summary ===")
for cc in corrupt_conditions:
    ie_dict = h2_results[cc]["ie"]
    cq_nie = ie_dict.get("cq_att_output", {}).get("nie", 0)
    total_nie = sum(v["nie"] for v in ie_dict.values()) if ie_dict else 1
    print(f"  {cc:>10s}:  CQ-Att NIE = {cq_nie:.4f}  ({cq_nie/max(total_nie,1e-6):.1%} of total NIE sum)")

print("\n=== H2 Key Findings ===")
te_c = h2_results["context"]["te_mean"]
te_q = h2_results["question"]["te_mean"]
print(f"  1. Context corruption TE ({te_c:.3f}) vs Question corruption TE ({te_q:.3f})")
print(f"     → {'Context' if te_c > te_q else 'Question'} corruption is more destructive ({max(te_c,te_q)/min(te_c,te_q):.2f}x)")
print(f"  2. CQ-Att as bottleneck: restoring CQ-Att output recovers most of the TE")
print(f"  3. Interaction: TE(both)={h2_results['both']['te_mean']:.3f} vs TE(C)+TE(Q)={te_c+te_q:.3f}"
      f"  → {'sub-additive (info overlap)' if h2_results['both']['te_mean'] < te_c+te_q else 'super-additive (synergy)'}")

In [ ]:
## H2 — Experiment 2: CQ Attention Ablation (Independent Bottleneck Validation)
##
## Zero out CQ Attention output at eval time and measure F1/EM drop.
## This provides independent validation of the CQ bottleneck hypothesis
## without relying on causal tracing (which uses probabilistic metrics).

import importlib, experiments.tracer
importlib.reload(experiments.tracer)
from experiments.tracer import qanet_forward
from EvaluateTools.eval_utils import convert_tokens, squad_evaluate

print("=" * 70)
print("H2 EXPERIMENT 2: CQ ATTENTION ABLATION")
print("=" * 70)
print("Methodology:")
print("  1. Run model normally (baseline)")
print("  2. Run model with CQ Attention output zeroed (skip_cq_att=True)")
print("  3. Compare F1/EM to quantify CQ Attention's contribution")
print("  4. Also zero Embedding Encoder outputs (C/Q separately) for comparison")
print(f"  Eval: full dev set ({len(dataset)} samples), batch_size=32\n")

loader_h2_abl = make_loader(dataset, batch_size=32, shuffle=False)

# We'll test multiple ablation conditions
ablation_configs = {
    "clean":       {"skip_cq_att": False},
    "skip_cq_att": {"skip_cq_att": True},
}

answers_all = {cfg: {} for cfg in ablation_configs}

with torch.no_grad():
    for Cwid, Ccid, Qwid, Qcid, y1, y2, ids in tqdm(loader_h2_abl, desc="CQ Ablation"):
        Cwid, Ccid = Cwid.to(DEVICE), Ccid.to(DEVICE)
        Qwid, Qcid = Qwid.to(DEVICE), Qcid.to(DEVICE)

        for cfg_name, kwargs in ablation_configs.items():
            p1, p2, _, _ = qanet_forward(model, Cwid, Ccid, Qwid, Qcid, **kwargs)
            p1_log = F.log_softmax(p1, dim=1)
            p2_log = F.log_softmax(p2, dim=1)
            outer = p1_log.unsqueeze(2) + p2_log.unsqueeze(1)
            mask_tri = torch.triu(torch.ones(outer.size(-2), outer.size(-1),
                                             device=DEVICE, dtype=torch.bool))
            outer = outer.masked_fill(~mask_tri, float('-inf'))
            flat = outer.view(outer.size(0), -1)
            idx = torch.argmax(flat, dim=1)
            L = p1.size(1)
            ymin = idx // L
            ymax = idx % L
            preds, _ = convert_tokens(eval_file, ids.tolist(), ymin.tolist(), ymax.tolist())
            answers_all[cfg_name].update(preds)

metrics_all = {}
for cfg_name in ablation_configs:
    metrics_all[cfg_name] = squad_evaluate(eval_file, answers_all[cfg_name])

print(f"\n{'='*70}")
print("CQ ATTENTION ABLATION RESULTS")
print(f"{'='*70}")
print(f"{'Condition':>15s}  {'F1':>7s}  {'EM':>7s}  {'ΔF1':>8s}  {'ΔEM':>8s}")
print("-" * 55)
base_f1 = metrics_all["clean"]["f1"]
base_em = metrics_all["clean"]["exact_match"]
for cfg_name in ablation_configs:
    m = metrics_all[cfg_name]
    df1 = m["f1"] - base_f1
    dem = m["exact_match"] - base_em
    print(f"{cfg_name:>15s}  {m['f1']:7.2f}  {m['exact_match']:7.2f}  {df1:+8.2f}  {dem:+8.2f}")

cq_f1_drop = base_f1 - metrics_all["skip_cq_att"]["f1"]
cq_em_drop = base_em - metrics_all["skip_cq_att"]["exact_match"]

print(f"\n--- Interpretation ---")
print(f"CQ Attention ablation causes:")
print(f"  F1 drop: {cq_f1_drop:.2f} ({cq_f1_drop/base_f1*100:.1f}% of baseline)")
print(f"  EM drop: {cq_em_drop:.2f} ({cq_em_drop/base_em*100:.1f}% of baseline)")
if cq_f1_drop > base_f1 * 0.5:
    print(f"  → CATASTROPHIC: removing CQ Attention destroys >{cq_f1_drop/base_f1*100:.0f}% of performance")
    print(f"  → Confirms CQ Attention as the critical information bottleneck")
    print(f"  → Consistent with causal tracing (Exp 1): CQ-Att restores majority of TE")
else:
    print(f"  → Moderate impact ({cq_f1_drop/base_f1*100:.1f}%): CQ Attention is important but not sole bottleneck")

print(f"\n--- Cross-validation with Causal Tracing ---")
print(f"  Causal tracing (Exp 1): CQ-Att NIE measures info FLOW through CQ attention")
print(f"  Ablation (this exp): CQ-Att ablation measures info STORED at CQ attention")
print(f"  Both methods should agree → CQ Attention is where C and Q streams merge")

print(f"\n--- Limitations ---")
print(f"  1. Zero ablation removes BOTH information content and activation magnitude")
print(f"  2. CQ resizer processes zeroed input → downstream effects may be indirect")
print(f"  3. This experiment confirms necessity but not sufficiency of CQ Attention")

# Save results
h2_abl_results = {
    cfg: {"f1": metrics_all[cfg]["f1"], "em": metrics_all[cfg]["exact_match"]}
    for cfg in ablation_configs
}
h2_abl_results["delta"] = {"f1": -cq_f1_drop, "em": -cq_em_drop}
os.makedirs("experiments/results/H2", exist_ok=True)
with open("experiments/results/H2/h2_cq_ablation.json", "w") as f:
    json.dump(h2_abl_results, f, indent=2)
print(f"\nSaved to experiments/results/H2/h2_cq_ablation.json")

In [ ]:
## H2 — Experiment 3: Selective Corruption (Answer vs Non-Answer Positions)
##
## Core question: Is context's causal contribution localized to the answer span,
## or distributed across all tokens? And does CQ Attention mediate both equally?
##
## This explains WHY question corruption may be more destructive:
## if context info is localized but question info is distributed,
## question has higher information density per token.

import importlib, experiments.tracer
importlib.reload(experiments.tracer)
from experiments.tracer import qanet_forward, RestoreSpec, compute_span_prob

NUM_H2_SEL = 200
NOISE_REPEATS_SEL = 3

random.seed(SEED); torch.manual_seed(SEED)
loader_h2_sel = make_loader(dataset, batch_size=1, shuffle=True)

print("=" * 70)
print("H2 EXPERIMENT 3: SELECTIVE CORRUPTION — INFORMATION LOCALIZATION")
print("=" * 70)
print("Conditions:")
print("  ans_only:     corrupt ONLY answer-span positions in context")
print("  non_ans_only: corrupt ONLY non-answer, non-padding positions in context")
print("  full_context: corrupt ALL context positions (baseline from Exp 1)")
print(f"Samples: {NUM_H2_SEL}, Noise repeats: {NOISE_REPEATS_SEL}")
print(f"Also measures CQ-Att IE for each condition (CQ recovery test)\n")

conditions = ["ans_only", "non_ans_only", "full_context"]
sel_te = {c: [] for c in conditions}
sel_cq_ie = {c: [] for c in conditions}
sel_n_corrupted = {c: [] for c in conditions}

cq_restore_spec = RestoreSpec(stage="cq_att", component="output")

samples_sel = 0
pbar_sel = tqdm(total=NUM_H2_SEL, desc="Selective Corruption")
with torch.no_grad():
    for Cwid, Ccid, Qwid, Qcid, y1, y2, ids in loader_h2_sel:
        if samples_sel >= NUM_H2_SEL:
            break
        Cwid, Ccid = Cwid.to(DEVICE), Ccid.to(DEVICE)
        Qwid, Qcid = Qwid.to(DEVICE), Qcid.to(DEVICE)
        y1, y2 = y1.to(DEVICE), y2.to(DEVICE)

        y1_val, y2_val = y1.item(), y2.item()
        L = Cwid.size(1)
        if y1_val >= L or y2_val >= L or y1_val > y2_val:
            continue

        p1_c, p2_c, clean_acts, _ = qanet_forward(
            model, Cwid, Ccid, Qwid, Qcid, collect=True)
        prob_clean = compute_span_prob(p1_c, p2_c, y1, y2)
        if prob_clean.item() < MIN_CLEAN_PROB:
            continue

        ans_mask = torch.zeros(1, L, device=DEVICE, dtype=torch.bool)
        ans_mask[0, y1_val:y2_val+1] = True
        non_pad = (Cwid != 0)
        non_ans_mask = (~ans_mask) & non_pad

        n_ans = ans_mask.sum().item()
        n_non_ans = non_ans_mask.sum().item()
        n_total = non_pad.sum().item()

        mask_map = {
            "ans_only": ans_mask,
            "non_ans_only": non_ans_mask,
            "full_context": None,
        }
        n_map = {
            "ans_only": n_ans,
            "non_ans_only": n_non_ans,
            "full_context": n_total,
        }

        for cond in conditions:
            reps_te, reps_cq_ie = [], []
            for rep in range(NOISE_REPEATS_SEL):
                ns = SEED + samples_sel * 100 + rep
                p1_x, p2_x, _, _ = qanet_forward(
                    model, Cwid, Ccid, Qwid, Qcid,
                    corrupt_target="context",
                    noise_std_scale=NOISE_STD_SCALE,
                    noise_seed=ns,
                    corrupt_mask_c=mask_map[cond],
                )
                prob_x = compute_span_prob(p1_x, p2_x, y1, y2)
                te = (prob_clean - prob_x).item()
                reps_te.append(te)

                if abs(te) < 1e-6:
                    continue

                p1_r, p2_r, _, _ = qanet_forward(
                    model, Cwid, Ccid, Qwid, Qcid,
                    corrupt_target="context",
                    noise_std_scale=NOISE_STD_SCALE,
                    noise_seed=ns,
                    corrupt_mask_c=mask_map[cond],
                    clean_acts=clean_acts,
                    restore_spec=cq_restore_spec,
                )
                prob_r = compute_span_prob(p1_r, p2_r, y1, y2)
                reps_cq_ie.append((prob_r - prob_x).item())

            sel_te[cond].append(np.mean(reps_te) if reps_te else 0)
            sel_cq_ie[cond].append(np.mean(reps_cq_ie) if reps_cq_ie else 0)
            sel_n_corrupted[cond].append(n_map[cond])

        samples_sel += 1
        pbar_sel.update(1)
        if samples_sel % 50 == 0:
            pbar_sel.set_postfix(
                TE_ans=f"{np.mean(sel_te['ans_only']):.3f}",
                TE_non=f"{np.mean(sel_te['non_ans_only']):.3f}",
                TE_full=f"{np.mean(sel_te['full_context']):.3f}",
            )
pbar_sel.close()

def _ci95(arr):
    a = np.array(arr)
    return 1.96 * a.std() / np.sqrt(len(a)) if len(a) > 1 else 0

print(f"\n{'='*70}")
print("SELECTIVE CORRUPTION RESULTS")
print(f"{'='*70}")
print(f"Samples: {samples_sel}, Noise repeats: {NOISE_REPEATS_SEL}\n")

print(f"--- Total Effect by Corruption Region ---")
print(f"{'Condition':>15s}  {'Mean TE':>10s}  {'±95%CI':>10s}  {'Avg #tokens':>12s}  {'TE/token':>10s}")
print("-" * 70)
for cond in conditions:
    te_arr = np.array(sel_te[cond])
    n_arr = np.array(sel_n_corrupted[cond])
    te_per_tok = te_arr.mean() / max(n_arr.mean(), 1) * 1000
    print(f"{cond:>15s}  {te_arr.mean():10.4f}  {_ci95(te_arr):10.4f}  "
          f"{n_arr.mean():12.1f}  {te_per_tok:10.4f}×10⁻³")

te_ans = np.mean(sel_te["ans_only"])
te_non = np.mean(sel_te["non_ans_only"])
te_full = np.mean(sel_te["full_context"])
n_ans_avg = np.mean(sel_n_corrupted["ans_only"])
n_non_avg = np.mean(sel_n_corrupted["non_ans_only"])

print(f"\n--- Information Localization Analysis ---")
print(f"TE(answer-only)  = {te_ans:.4f}  (avg {n_ans_avg:.1f} tokens)")
print(f"TE(non-answer)   = {te_non:.4f}  (avg {n_non_avg:.1f} tokens)")
print(f"TE(full context) = {te_full:.4f}  (avg {np.mean(sel_n_corrupted['full_context']):.1f} tokens)")
print(f"TE(ans) + TE(non) = {te_ans + te_non:.4f}  vs  TE(full) = {te_full:.4f}")
overlap = te_full - (te_ans + te_non)
print(f"Interaction = {overlap:.4f} ({'sub-additive' if overlap < 0 else 'super-additive'})")

te_per_tok_ans = te_ans / max(n_ans_avg, 1)
te_per_tok_non = te_non / max(n_non_avg, 1)
density_ratio = te_per_tok_ans / max(te_per_tok_non, 1e-8)
print(f"\nInformation density (TE per token):")
print(f"  Answer tokens:     {te_per_tok_ans:.6f}")
print(f"  Non-answer tokens: {te_per_tok_non:.6f}")
print(f"  Density ratio: {density_ratio:.1f}x")
if density_ratio > 3:
    print(f"  → Answer tokens are {density_ratio:.0f}x more information-dense than non-answer tokens")
    print(f"  → Context's causal contribution is highly LOCALIZED to the answer span")
elif density_ratio > 1.5:
    print(f"  → Answer tokens are moderately more important ({density_ratio:.1f}x)")
else:
    print(f"  → Context information is relatively distributed")

print(f"\n--- CQ Attention Recovery by Condition ---")
print(f"{'Condition':>15s}  {'Mean CQ IE':>10s}  {'NIE (% of TE)':>15s}")
print("-" * 50)
for cond in conditions:
    cq_ie_mean = np.mean(sel_cq_ie[cond])
    te_mean = np.mean(sel_te[cond])
    nie = cq_ie_mean / max(abs(te_mean), 1e-8) * 100
    print(f"{cond:>15s}  {cq_ie_mean:10.4f}  {nie:14.1f}%")

print(f"\n--- Synthesis: Why Question Corruption May Be More Destructive ---")
print(f"If context info is localized to ~{n_ans_avg:.0f} answer tokens but question")
print(f"info is distributed across all ~{Qwid.size(1):.0f} question tokens:")
print(f"  → Question has higher global information density")
print(f"  → Corrupting ANY Q token degrades the entire query signal")
print(f"  → Corrupting non-answer C tokens barely affects performance")
print(f"  → This explains the TE asymmetry observed in Experiment 1")

print(f"\n--- Limitations ---")
print(f"  1. Answer span corruption DIRECTLY removes the target signal — high TE is expected")
print(f"  2. The interesting finding is whether non-answer corruption is near zero or not")
print(f"  3. Noise-based corruption may not perfectly isolate positional information")
print(f"  4. CQ Attention recovery % may vary with corruption locality")

sel_results = {}
for cond in conditions:
    sel_results[cond] = {
        "te_mean": float(np.mean(sel_te[cond])),
        "te_ci95": float(_ci95(sel_te[cond])),
        "cq_ie_mean": float(np.mean(sel_cq_ie[cond])),
        "cq_nie": float(np.mean(sel_cq_ie[cond]) / max(abs(np.mean(sel_te[cond])), 1e-8)),
        "avg_tokens_corrupted": float(np.mean(sel_n_corrupted[cond])),
    }
sel_results["info_density_ratio"] = float(density_ratio)
with open("experiments/results/H2/h2_selective_corruption.json", "w") as f:
    json.dump(sel_results, f, indent=2)
print(f"\nSaved to experiments/results/H2/h2_selective_corruption.json")

### H3: Pointer Layer Asymmetric Connections & Pass Role Differentiation

**Hypothesis**: The asymmetric M1/M2 → start, M1/M3 → end is better than all symmetric alternatives. M2 and M3 encode distinct temporal features (M2 specializes in start-boundary, M3 in end-boundary).

**Method**: (A) Replace M1/M2/M3 wiring at the Pointer layer and measure F1/EM. (B) CKA & cosine similarity analysis.

In [ ]:
## H3 Phase A — Pointer Replacement Experiments
from experiments.run_H3 import POINTER_CONFIGS, pointer_forward, run_phase_a, run_phase_b
from experiments.run_H3 import cosine_sim_per_token, linear_cka
from EvaluateTools.eval_utils import convert_tokens, squad_evaluate

print("=" * 70)
print("H3 PHASE A: POINTER WIRING REPLACEMENT EXPERIMENTS")
print("=" * 70)
print("Original design: P(start) = softmax(w1·[M1;M2]), P(end) = softmax(w2·[M1;M3])")
print("Configs tested:")
print("  original  — M1+M2→start, M1+M3→end  (paper design)")
print("  swap      — M1+M3→start, M1+M2→end  (swap M2/M3)")
print("  sym_M2    — M1+M2→start, M1+M2→end  (use M2 for both)")
print("  sym_M3    — M1+M3→start, M1+M3→end  (use M3 for both)")
print("  only_M1   — M1→start, M1→end         (remove M2/M3)")
print("  no_M1     — M2→start, M3→end         (remove M1)")
print(f"\nEval on full dev set ({len(dataset)} samples), batch_size=32.\n")

phase_a = run_phase_a(model, dataset, eval_file, batch_size=32)

os.makedirs("experiments/results/H3", exist_ok=True)

orig_f1 = phase_a["original"]["f1"]
orig_em = phase_a["original"]["em"]

print(f"\n{'='*70}")
print("PHASE A RESULTS")
print(f"{'='*70}")
print(f"{'Config':>12s}  {'F1':>7s}  {'EM':>7s}  {'ΔF1':>8s}  {'ΔEM':>8s}  {'%F1 lost':>9s}")
print("-" * 65)
for cfg, res in sorted(phase_a.items(), key=lambda x: -x[1]["f1"]):
    df1 = res["f1"] - orig_f1
    dem = res["em"] - orig_em
    pct = abs(df1) / orig_f1 * 100 if df1 != 0 else 0
    marker = " ◀ baseline" if cfg == "original" else ""
    print(f"{cfg:>12s}  {res['f1']:7.2f}  {res['em']:7.2f}  {df1:+8.2f}  {dem:+8.2f}  {pct:8.1f}%{marker}")

print(f"\n--- Key Observations ---")
swap_drop = orig_f1 - phase_a.get("swap", {}).get("f1", orig_f1)
sym2_drop = orig_f1 - phase_a.get("sym_M2", {}).get("f1", orig_f1)
sym3_drop = orig_f1 - phase_a.get("sym_M3", {}).get("f1", orig_f1)
only_m1_drop = orig_f1 - phase_a.get("only_M1", {}).get("f1", orig_f1)
no_m1_drop = orig_f1 - phase_a.get("no_M1", {}).get("f1", orig_f1)
print(f"  1. swap (M2↔M3): ΔF1 = {-swap_drop:+.2f} → M2/M3 are NOT interchangeable")
print(f"  2. sym_M2 vs sym_M3: ΔF1 = {-sym2_drop:+.2f} vs {-sym3_drop:+.2f} → {'M2 more suitable for both' if sym2_drop < sym3_drop else 'M3 more suitable for both'}")
print(f"  3. only_M1: ΔF1 = {-only_m1_drop:+.2f} → M1 alone loses {only_m1_drop:.1f} F1 (M2/M3 contribute)")
print(f"  4. no_M1:   ΔF1 = {-no_m1_drop:+.2f} → Removing M1 loses {no_m1_drop:.1f} F1 (M1 contributes)")
print(f"  5. Asymmetric gain = swap_drop ({swap_drop:.2f}) vs best symmetric ({min(sym2_drop, sym3_drop):.2f})"
      f" → asymmetric wiring {'justified' if swap_drop > 0 else 'not clearly better'}")

In [ ]:
## H3 Phase B — Representation Similarity Analysis
import importlib, experiments.run_H3
importlib.reload(experiments.run_H3)
from experiments.run_H3 import run_phase_b

print("=" * 70)
print("H3 PHASE B: REPRESENTATION SIMILARITY ANALYSIS (M1/M2/M3)")
print("=" * 70)
print("Metrics: Cosine Similarity (per-token, then averaged) + CKA (dataset-level)")
print("CKA = Centered Kernel Alignment — measures representational similarity")
print("  CKA=1 → identical representations, CKA=0 → orthogonal representations\n")

phase_b = run_phase_b(model, dataset, num_samples=1000, batch_size=32, seed=42)

with open("experiments/results/H3/h3_results.json", "w") as f:
    json.dump({"phase_a": phase_a, "phase_b": phase_b}, f, indent=2)

print(f"\n{'='*70}")
print("PHASE B RESULTS")
print(f"{'='*70}")

print(f"\n--- Global Cosine Similarity (all tokens averaged) ---")
print(f"{'Pair':>10s}  {'Mean':>8s}  {'±95%CI':>8s}  {'Interpretation'}")
print("-" * 55)
for pair, stats in phase_b["global_cosine"].items():
    sim = stats["mean"]
    interp = "very high" if sim > 0.99 else ("high" if sim > 0.95 else ("moderate" if sim > 0.8 else "low"))
    print(f"{pair:>10s}  {sim:8.4f}  {stats['ci95']:8.4f}  ({interp} similarity)")

gc = phase_b["global_cosine"]
m12 = gc.get("M1_M2", {}).get("mean", 0)
m13 = gc.get("M1_M3", {}).get("mean", 0)
m23 = gc.get("M2_M3", {}).get("mean", 0)
print(f"\nOrdering: M2_M3 ({m23:.4f}) > M1_M2 ({m12:.4f}) > M1_M3 ({m13:.4f})")
print(f"→ M2 and M3 are most similar to each other (consecutive passes), M1 diverges most from M3")

print(f"\n--- M2 vs M3 Cosine Similarity by Answer Position ---")
print(f"{'Region':>20s}  {'Mean':>8s}  {'±95%CI':>8s}")
print("-" * 45)
pos_data = phase_b.get("position_cosine_M2_M3", {})
for region in ["answer_start", "answer_interior", "answer_end", "non_answer"]:
    s = pos_data.get(region, {})
    print(f"{region:>20s}  {s.get('mean',0):8.4f}  {s.get('ci95',0):8.4f}")

ans_start_sim = pos_data.get("answer_start", {}).get("mean", 0)
ans_end_sim = pos_data.get("answer_end", {}).get("mean", 0)
non_ans_sim = pos_data.get("non_answer", {}).get("mean", 0)
print(f"\nAnswer boundary vs non-answer difference:")
print(f"  start:      {ans_start_sim:.4f} - {non_ans_sim:.4f} = {ans_start_sim - non_ans_sim:+.4f}")
print(f"  end:        {ans_end_sim:.4f} - {non_ans_sim:.4f} = {ans_end_sim - non_ans_sim:+.4f}")
if ans_start_sim < non_ans_sim or ans_end_sim < non_ans_sim:
    print(f"  → Answer positions show LOWER M2-M3 similarity → M2/M3 diverge at answer boundaries")
else:
    print(f"  → Answer positions show similar/higher M2-M3 similarity")

print(f"\n--- CKA (Centered Kernel Alignment) ---")
cka = phase_b.get("cka", {})
print(f"{'Pair':>10s}  {'CKA':>8s}  {'Interpretation'}")
print("-" * 45)
for pair in ["M1_M2", "M1_M3", "M2_M3"]:
    v = cka.get(pair, 0)
    interp = "nearly identical" if v > 0.95 else ("high" if v > 0.85 else ("moderate" if v > 0.7 else "low"))
    print(f"{pair:>10s}  {v:8.4f}  ({interp})")

cka_12 = cka.get("M1_M2", 0)
cka_13 = cka.get("M1_M3", 0)
cka_23 = cka.get("M2_M3", 0)
print(f"\nCKA ordering: M2_M3 ({cka_23:.4f}) > M1_M2 ({cka_12:.4f}) > M1_M3 ({cka_13:.4f})")
print(f"CKA drop from M1→M2→M3: each pass transforms representations further from M1")
print(f"  M1→M2 change: 1−CKA = {1-cka_12:.4f}")
print(f"  M2→M3 change: 1−CKA = {1-cka_23:.4f}")
print(f"  M1→M3 change: 1−CKA = {1-cka_13:.4f}")

print(f"\n--- H3 Phase B Key Findings ---")
print(f"  1. M2 & M3 share high but not identical representations (CKA={cka_23:.3f} < 1)")
print(f"  2. Each encoder pass progressively transforms: M1→M2→M3 (CKA decreases with distance)")
print(f"  3. The asymmetric Pointer design exploits the M2/M3 divergence")
print(f"  4. Combined with Phase A: swap causes {swap_drop:.2f} F1 drop → the divergence is task-relevant")

In [ ]:
## H3 — Visualization
import matplotlib.pyplot as plt
import seaborn as sns

print("Generating H3 visualizations: 3 plots")
print("  1. Phase A: Pointer wiring configs — F1/EM bar chart")
print("  2. Phase B: CKA similarity matrix (M1/M2/M3)")
print("  3. Phase B: M2 vs M3 cosine similarity by answer position\n")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- 1. Phase A: F1/EM bar chart ---
ax = axes[0]
cfgs = sorted(phase_a.keys(), key=lambda c: -phase_a[c]["f1"])
f1s = [phase_a[c]["f1"] for c in cfgs]
ems = [phase_a[c]["em"] for c in cfgs]
x = np.arange(len(cfgs))
w = 0.35
ax.bar(x - w/2, f1s, w, label="F1", color="#2196F3", edgecolor="black")
ax.bar(x + w/2, ems, w, label="EM", color="#FF9800", edgecolor="black")
ax.set_xticks(x); ax.set_xticklabels(cfgs, rotation=30, ha="right")
ax.set_ylabel("Score"); ax.set_title("H3-A: Pointer Wiring Configs")
ax.legend(); ax.axhline(phase_a["original"]["f1"], color="blue", ls="--", alpha=0.3)

# --- 2. Phase B: CKA matrix ---
ax = axes[1]
pairs_order = ["M1", "M2", "M3"]
cka_mat = np.eye(3)
cka = phase_b.get("cka", {})
pair_map = {("M1","M2"): cka.get("M1_M2", 0), ("M1","M3"): cka.get("M1_M3", 0), ("M2","M3"): cka.get("M2_M3", 0)}
for (a,b), v in pair_map.items():
    i, j = pairs_order.index(a), pairs_order.index(b)
    cka_mat[i,j] = cka_mat[j,i] = v
sns.heatmap(cka_mat, xticklabels=pairs_order, yticklabels=pairs_order,
            annot=True, fmt=".3f", cmap="YlOrRd", vmin=0, vmax=1, ax=ax)
ax.set_title("H3-B: CKA Similarity")

# --- 3. Phase B: Position-stratified cosine ---
ax = axes[2]
pos_data = phase_b.get("position_cosine_M2_M3", {})
regions = ["answer_start", "answer_interior", "answer_end", "non_answer"]
means_pos = [pos_data.get(r, {}).get("mean", 0) for r in regions]
errs_pos = [pos_data.get(r, {}).get("ci95", 0) for r in regions]
colors_pos = ["#4CAF50", "#8BC34A", "#FF9800", "#9E9E9E"]
ax.bar(["Start", "Interior", "End", "Non-answer"], means_pos, yerr=errs_pos,
       color=colors_pos, capsize=5, edgecolor="black")
ax.set_ylabel("Cosine Sim (M2 vs M3)")
ax.set_title("H3-B: M2 vs M3 by Position")

fig.suptitle("H3: Pointer Asymmetry & Pass Role Differentiation", fontsize=14, fontweight="bold")
fig.tight_layout(); plt.show()

In [ ]:
## H3 — Experiment C: Linear Probe for Start/End Token Specialization
##
## Core claim to verify: "M2 specializes in start-boundary, M3 in end-boundary"
## Phase A (wiring ablation) shows SWAP hurts performance, and Phase B (CKA/cosine)
## shows M2/M3 are different — but neither proves M2 encodes START and M3 encodes END.
##
## This experiment trains linear probes directly:
##   M2 tokens → predict is_start_token (ROC-AUC)
##   M2 tokens → predict is_end_token
##   M3 tokens → predict is_start_token
##   M3 tokens → predict is_end_token
## If AUC(M2→start) > AUC(M3→start) and AUC(M3→end) > AUC(M2→end) → direct evidence

import importlib, experiments.tracer
importlib.reload(experiments.tracer)
from experiments.tracer import qanet_forward
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

N_PROBE_H3 = 500
random.seed(42); torch.manual_seed(42)
loader_h3_probe = make_loader(dataset, batch_size=1, shuffle=True)

print("=" * 70)
print("H3 EXPERIMENT C: LINEAR PROBE — START/END SPECIALIZATION")
print("=" * 70)
print("Methodology:")
print("  1. Collect M1/M2/M3 token representations for each sample")
print("  2. Labels: is_start (token == answer_start), is_end (token == answer_end)")
print("  3. Train LogisticRegression probes (balanced class weight)")
print("  4. Compare ROC-AUC across M1/M2/M3 for each label type")
print(f"  Samples: {N_PROBE_H3}, 80/20 train/test split\n")

probe_feats = {"M1": {"X": [], "y_start": [], "y_end": []},
               "M2": {"X": [], "y_start": [], "y_end": []},
               "M3": {"X": [], "y_start": [], "y_end": []}}

samples_h3p = 0
with torch.no_grad():
    for Cwid, Ccid, Qwid, Qcid, y1, y2, ids in tqdm(
            loader_h3_probe, total=N_PROBE_H3, desc="H3 Probe features"):
        if samples_h3p >= N_PROBE_H3:
            break
        y1_val, y2_val = y1.item(), y2.item()
        L = Cwid.size(1)
        if y1_val >= L or y2_val >= L or y1_val > y2_val:
            continue

        Cwid, Ccid = Cwid.to(DEVICE), Ccid.to(DEVICE)
        Qwid, Qcid = Qwid.to(DEVICE), Qcid.to(DEVICE)

        _, _, _, intermediates = qanet_forward(
            model, Cwid, Ccid, Qwid, Qcid, collect=False)
        M1, M2, M3 = intermediates["M1"], intermediates["M2"], intermediates["M3"]

        pad_mask = (Cwid[0].cpu() != 0).numpy()
        label_start = np.zeros(L, dtype=int)
        label_end = np.zeros(L, dtype=int)
        label_start[y1_val] = 1
        label_end[y2_val] = 1

        for name, M in [("M1", M1), ("M2", M2), ("M3", M3)]:
            feat = M[0].cpu().float().numpy()  # [d_model, L]
            for t in range(L):
                if not pad_mask[t]:
                    continue
                probe_feats[name]["X"].append(feat[:, t])
                probe_feats[name]["y_start"].append(label_start[t])
                probe_feats[name]["y_end"].append(label_end[t])

        samples_h3p += 1

print(f"\nCollected {samples_h3p} samples")
for name in ["M1", "M2", "M3"]:
    n_total = len(probe_feats[name]["X"])
    n_start = sum(probe_feats[name]["y_start"])
    n_end = sum(probe_feats[name]["y_end"])
    print(f"  {name}: {n_total} tokens, {n_start} start+ ({n_start/n_total*100:.2f}%), "
          f"{n_end} end+ ({n_end/n_total*100:.2f}%)")

print(f"\n{'='*70}")
print("LINEAR PROBE RESULTS — START/END TOKEN PREDICTION")
print(f"{'='*70}")

probe_results_h3 = {}
print(f"\n{'Repr':>4s}  {'Target':>8s}  {'AUC':>6s}  {'Acc':>6s}  {'F1(+)':>7s}  {'N_test':>7s}")
print("-" * 50)

for name in ["M1", "M2", "M3"]:
    X_all = np.array(probe_feats[name]["X"])
    y_start_all = np.array(probe_feats[name]["y_start"])
    y_end_all = np.array(probe_feats[name]["y_end"])

    n = len(X_all)
    idx = np.arange(n)
    np.random.seed(42)
    np.random.shuffle(idx)
    split = int(0.8 * n)

    for target_name, y_all in [("start", y_start_all), ("end", y_end_all)]:
        X_tr, X_te = X_all[idx[:split]], X_all[idx[split:]]
        y_tr, y_te = y_all[idx[:split]], y_all[idx[split:]]

        if y_tr.sum() < 5 or y_te.sum() < 5:
            print(f"{name:>4s}  {target_name:>8s}  (insufficient positives)")
            continue

        clf = LogisticRegression(max_iter=2000, class_weight="balanced", C=1.0, solver="lbfgs")
        clf.fit(X_tr, y_tr)
        y_prob = clf.predict_proba(X_te)[:, 1]
        y_pred = clf.predict(X_te)

        auc = roc_auc_score(y_te, y_prob)
        acc = accuracy_score(y_te, y_pred)
        f1 = f1_score(y_te, y_pred)

        probe_results_h3[(name, target_name)] = {"auc": auc, "acc": acc, "f1": f1}
        print(f"{name:>4s}  {target_name:>8s}  {auc:6.3f}  {acc:6.3f}  {f1:7.3f}  {len(y_te):7d}")
    print()

print(f"--- Specialization Analysis ---")
auc_m2_start = probe_results_h3.get(("M2", "start"), {}).get("auc", 0)
auc_m3_start = probe_results_h3.get(("M3", "start"), {}).get("auc", 0)
auc_m2_end = probe_results_h3.get(("M2", "end"), {}).get("auc", 0)
auc_m3_end = probe_results_h3.get(("M3", "end"), {}).get("auc", 0)
auc_m1_start = probe_results_h3.get(("M1", "start"), {}).get("auc", 0)
auc_m1_end = probe_results_h3.get(("M1", "end"), {}).get("auc", 0)

print(f"\nStart token prediction:")
print(f"  M1: AUC={auc_m1_start:.3f}  M2: AUC={auc_m2_start:.3f}  M3: AUC={auc_m3_start:.3f}")
print(f"  Best predictor: {'M2' if auc_m2_start > auc_m3_start else 'M3'} "
      f"(Δ = {abs(auc_m2_start - auc_m3_start):.3f})")

print(f"\nEnd token prediction:")
print(f"  M1: AUC={auc_m1_end:.3f}  M2: AUC={auc_m2_end:.3f}  M3: AUC={auc_m3_end:.3f}")
print(f"  Best predictor: {'M3' if auc_m3_end > auc_m2_end else 'M2'} "
      f"(Δ = {abs(auc_m3_end - auc_m2_end):.3f})")

m2_start_adv = auc_m2_start - auc_m3_start
m3_end_adv = auc_m3_end - auc_m2_end

print(f"\nSpecialization pattern:")
print(f"  M2 start advantage: {m2_start_adv:+.3f} AUC")
print(f"  M3 end advantage:   {m3_end_adv:+.3f} AUC")

if m2_start_adv > 0.01 and m3_end_adv > 0.01:
    print(f"  → CONFIRMED: M2 specializes in start boundary, M3 in end boundary")
    print(f"  → This provides direct causal evidence for the asymmetric Pointer design")
elif m2_start_adv > 0 and m3_end_adv > 0:
    print(f"  → WEAK EVIDENCE: direction consistent but effect size is small")
    print(f"  → Specialization exists but may not be the primary factor")
else:
    print(f"  → NOT CONFIRMED: the expected specialization pattern is not observed")
    print(f"  → Asymmetric wiring may work for reasons other than start/end specialization")

print(f"\n--- Progressive Encoding Analysis ---")
print(f"  M1→M2→M3 start AUC: {auc_m1_start:.3f} → {auc_m2_start:.3f} → {auc_m3_start:.3f}")
print(f"  M1→M2→M3 end AUC:   {auc_m1_end:.3f} → {auc_m2_end:.3f} → {auc_m3_end:.3f}")
if auc_m3_start > auc_m1_start and auc_m3_end > auc_m1_end:
    print(f"  → Later passes encode BOTH boundaries better (progressive refinement)")
elif auc_m3_start < auc_m1_start:
    print(f"  → Start prediction degrades from M1→M3 → M3 loses start info (divergence)")

print(f"\n--- Limitations ---")
print(f"  1. Linear probes only detect linearly decodable information")
print(f"  2. Extreme class imbalance (~{samples_h3p}/{len(probe_feats['M1']['X']):.0f} ≈ "
      f"{samples_h3p/len(probe_feats['M1']['X'])*100:.2f}% positive)")
print(f"  3. Specialization may exist in non-linear subspaces not captured here")
print(f"  4. The Pointer layer uses bilinear projection, not linear readout")

os.makedirs("experiments/results/H3", exist_ok=True)
h3_probe_save = {f"{k[0]}_{k[1]}": v for k, v in probe_results_h3.items()}
h3_probe_save["specialization"] = {
    "m2_start_advantage": float(m2_start_adv),
    "m3_end_advantage": float(m3_end_adv),
}
with open("experiments/results/H3/h3_linear_probe.json", "w") as f:
    json.dump(h3_probe_save, f, indent=2)
print(f"\nSaved to experiments/results/H3/h3_linear_probe.json")

### Stage 3 — Results Summary

All results are saved to `experiments/results/`:
- `H1/h1_results.json` — Component-level causal tracing AIE values
- `H2/h2_results.json` — Dual-stream TE & NIE per corruption target
- `H3/h3_results.json` — Pointer replacement F1/EM & representation similarity

In [ ]:
## Stage 3 — Summary printout (reads from saved JSON files)
import json

print("=" * 70)
print("STAGE 3: EXPERIMENTAL INVESTIGATION — FINAL SUMMARY")
print("=" * 70)

# ---------- H1 ----------
with open("experiments/results/H1/h1_results.json") as f:
    h1_data = json.load(f)
h1_res = h1_data["results"]

h1_global_path = "experiments/results/H1/h1_ablation_global.json"
h1_block_path = "experiments/results/H1/h1_ablation_perblock.json"
has_m1 = os.path.exists(h1_global_path)
has_m3 = os.path.exists(h1_block_path)

print(f"\n{'─'*70}")
print("[H1] Conv vs Self-Attention Causal Importance — Three-Method Cross-Validation")
print(f"{'─'*70}")
print(f"Hypothesis: Conv > Self-Attn in causal importance, concentrated in shallow blocks & early passes.")
print(f"\n  Method 2 (ROME Causal Tracing): {h1_data['meta']['num_samples']} samples, Avg TE = {h1_data['meta']['avg_te']:.4f}")

conv_aie = sum(h1_res[k]["aie_span"] for k in h1_res if "conv" in k)
attn_aie = sum(h1_res[k]["aie_span"] for k in h1_res if k.endswith("self_attn"))
ffn_aie = sum(h1_res[k]["aie_span"] for k in h1_res if k.endswith("ffn"))
total_aie = conv_aie + attn_aie + ffn_aie
print(f"    Total AIE: Conv = {conv_aie:.4f} ({conv_aie/total_aie:.1%}), Attn = {attn_aie:.4f} ({attn_aie/total_aie:.1%}), FFN = {ffn_aie:.4f} ({ffn_aie/total_aie:.1%})")

print(f"  Top 5 sub-layers:")
for k, v in sorted(h1_res.items(), key=lambda x: -x[1]["aie_span"])[:5]:
    print(f"    {k:<25s}  AIE = {v['aie_span']:.4f} ± {v['ci95']:.4f}")

if has_m1:
    with open(h1_global_path) as f:
        h1_glob = json.load(f)
    base = h1_glob["baseline"]["f1"]
    print(f"\n  Method 1 (Global Ablation): baseline F1 = {base:.2f}")
    for k in ["skip_conv", "skip_attn", "skip_ffn"]:
        r = h1_glob[k]
        print(f"    {k:>14s}: F1 = {r['f1']:.2f} (ΔF1 = {r['f1']-base:+.2f})")

if has_m3:
    with open(h1_block_path) as f:
        h1_blk = json.load(f)
    conv_drops_h1 = [base - r["f1"] for k, r in h1_blk.items() if k.endswith("_conv")]
    attn_drops_h1 = [base - r["f1"] for k, r in h1_blk.items() if k.endswith("_self_attn")]
    print(f"\n  Method 3 (Per-block Ablation): 42 configs")
    print(f"    Avg F1 drop — Conv: {np.mean(conv_drops_h1):.2f}, Attn: {np.mean(attn_drops_h1):.2f}")
    print(f"    Conv/Attn ratio: {np.mean(conv_drops_h1)/max(np.mean(attn_drops_h1),0.01):.2f}x")

all_agree = conv_aie > attn_aie
if has_m1:
    all_agree = all_agree and (h1_glob["skip_conv"]["f1"] < h1_glob["skip_attn"]["f1"])
if has_m3:
    all_agree = all_agree and (np.mean(conv_drops_h1) > np.mean(attn_drops_h1))
print(f"\n  ★ VERDICT: All methods agree Conv > Attn? {'YES' if all_agree else 'PARTIAL'}")

# ---------- H2 ----------
with open("experiments/results/H2/h2_results.json") as f:
    h2_res = json.load(f)

print(f"\n{'─'*70}")
print("[H2] Context vs Question Dual-Stream Information Flow")
print(f"{'─'*70}")
print(f"Hypothesis: CQ Attention is the critical information bottleneck (recovery >70%).")

for cc in ["context", "question", "both"]:
    te = h2_res[cc]["te_mean"]
    ci = h2_res[cc]["te_ci95"]
    cq = h2_res[cc]["ie"].get("cq_att_output", {})
    cq_nie = cq.get("nie", 0) if cq else 0
    print(f"  {cc:>10s}: TE = {te:.4f} ± {ci:.4f}, CQ-Att recovery = {cq_nie:.1%}")

te_c = h2_res["context"]["te_mean"]
te_q = h2_res["question"]["te_mean"]
te_b = h2_res["both"]["te_mean"]
print(f"\n  TE additivity: TE(C)+TE(Q) = {te_c+te_q:.4f} vs TE(both) = {te_b:.4f}")
print(f"  → {'Sub-additive (information overlap)' if te_b < te_c+te_q else 'Super-additive (synergy)'}")
all_cq_above = all(
    h2_res[cc]["ie"].get("cq_att_output", {}).get("nie", 0) > 0.7
    for cc in ["context", "question", "both"]
)
print(f"\n  ★ VERDICT: CQ-Att bottleneck (>70% recovery)? {'YES' if all_cq_above else 'PARTIAL'}")

# ---------- H3 ----------
with open("experiments/results/H3/h3_results.json") as f:
    h3_data = json.load(f)

print(f"\n{'─'*70}")
print("[H3] Pointer Layer Asymmetric Connections & Pass Role Differentiation")
print(f"{'─'*70}")
print(f"Hypothesis: Asymmetric M1/M2→start, M1/M3→end is optimal. M2/M3 encode distinct features.")

pa = h3_data["phase_a"]
orig_f1_h3 = pa["original"]["f1"]
orig_em_h3 = pa["original"]["em"]
print(f"\n  Phase A — Pointer Wiring Replacement (baseline F1={orig_f1_h3:.2f}, EM={orig_em_h3:.2f}):")
for cfg in ["original", "swap", "sym_M2", "sym_M3", "only_M1", "no_M1"]:
    if cfg in pa:
        r = pa[cfg]
        df = r["f1"] - orig_f1_h3
        marker = " ◀ baseline" if cfg == "original" else ""
        print(f"    {cfg:>12s}: F1={r['f1']:.2f} (ΔF1={df:+.2f}), EM={r['em']:.2f}{marker}")

pb = h3_data["phase_b"]
cka = pb.get("cka", {})
print(f"\n  Phase B — Representation Similarity:")
for pair in ["M1_M2", "M1_M3", "M2_M3"]:
    v = cka.get(pair, 0)
    print(f"    CKA({pair}) = {v:.4f}")

gc = pb.get("global_cosine", {})
for pair in ["M1_M2", "M1_M3", "M2_M3"]:
    s = gc.get(pair, {})
    print(f"    Cosine({pair}) = {s.get('mean',0):.4f} ± {s.get('ci95',0):.4f}")

swap_ok = pa.get("swap", {}).get("f1", orig_f1_h3) < orig_f1_h3
orig_best = all(pa[cfg]["f1"] <= orig_f1_h3 for cfg in pa if cfg != "original")
print(f"\n  ★ VERDICT: Original wiring is optimal? {'YES' if orig_best else 'NO'}")
print(f"  ★ VERDICT: M2/M3 encode distinct features? {'YES (CKA<1, swap hurts)' if swap_ok and cka.get('M2_M3',1) < 0.99 else 'PARTIAL'}")

print(f"\n{'='*70}")
print("All results saved to experiments/results/{H1,H2,H3}/")
print("=" * 70)